# Production RAG on Cloud SQL + pgvector — v2

This replaces `rag_end_to_end_psycopg2_new_v1.1.ipynb`. v1.1 was a *method bake-off*; this is the
**pipeline you would actually deploy**, built the way retrieval systems are built in production.

## Why v1.1 answered the satisfaction question wrong

You asked *"What was the Customer Satisfaction level for supplier performance review?"* and got
`Overall district satisfaction was 8.5/10` instead of `Customer satisfaction: 4.8/5 stars`.
Four independent defects caused that, and every one of them gets worse on real data:

| # | Defect in v1.1 | Consequence |
|---|---|---|
| 1 | **Chunks were stored naked** — Step 8 saved only `doc_id, chunk_index, content`. All the metadata written in Step 7a (supplier, category, date, doc type) was discarded before embedding. | A chunk reading *"Customer satisfaction: 4.8/5 stars"* has no supplier attached, so neither the retriever nor the LLM can tell whose score it is. |
| 2 | **Pure dense retrieval, no keyword leg.** | Exact tokens — `BID-2024-089`, `4.8/5`, `Net-45`, a SKU — are precisely where embeddings are weakest. Dense-only silently misses ID lookups. |
| 3 | **The generation prompt had no disambiguation or citation rules.** | The document holds *two* satisfaction numbers (per-supplier `4.8/5`, district-wide `8.5/10`). The model picked one, unlabelled and uncited, and you had no way to see it chose wrong. |
| 4 | **No reranking, `TOP_K = 3`.** | Whatever the first-stage vector search returns *is* the answer. One weak embedding = one wrong answer. |

Two further things in v1.1 would have hurt you at deploy time:

- **The leaderboard measured the wrong thing.** Latency included the ~3.2 s Vertex embedding round-trip,
  which swamped the actual Postgres search (sub-millisecond on 20 rows). Every index scored ~3.2 s, so
  the `hnsw` vs `ivfflat` vs `none` comparison carried no signal — and it crowned **`none`
  (brute-force sequential scan)** the winner. Ship that and every query becomes a full table scan.
- **The gold set had a wrong label.** `"Which supplier offered an early payment discount?" → globex-dairy-2024`
  — but Sysco (1%) and US Foods (2%) offer one too. The benchmark was scoring correct retrievals as failures.

## What v2 does instead

1. **Contextual chunk headers** — every chunk is embedded with its document title + supplier + category + date prepended (a cheap form of Anthropic's *contextual retrieval*). Fixes defect 1.
2. **Hybrid retrieval** — pgvector KNN **+** Postgres full-text search, fused with **Reciprocal Rank Fusion** in one SQL round trip. Fixes defect 2.
3. **Reranking** — a second-stage precision pass over ~20 candidates before the LLM sees anything. Fixes defect 4.
4. **A grounded prompt** — mandatory citations, "list *all* matching entities", "answer for the entity that was asked about", fixed refusal string. Fixes defect 3.
5. **A real schema** — `rag_documents` / `rag_chunks`, FK cascade, JSONB metadata, generated `tsvector`, HNSW + GIN indexes, and content-hash **incremental re-indexing** (no more re-embedding everything on every run).
6. **`gemini-embedding-001` truncated to 1536 dims** via Matryoshka (MRL) — near-full quality, and it fits under pgvector's 2000-dim index ceiling, so you get **HNSW instead of brute force**.
7. **Honest evaluation** — 13 questions including exact-ID lookups and an unanswerable one; recall@k, MRR, nDCG, *answer-support*, refusal correctness, and latency split per stage.
8. **Ops hygiene** — connection pooling, retry with backoff, secrets from the environment, and no `DROP TABLE` of your production index.

> **Security — action required.** v1.1 carries a live Postgres password in cleartext in cell 2. If that
> file has been shared or committed anywhere, **rotate that credential.** v2 reads it from `$PGPASSWORD`
> and prompts if unset.

Run cells top to bottom.

## New to RAG? Read this first (5 minutes)

**RAG = Retrieval-Augmented Generation.** An LLM cannot know what is in *your* private documents, and
if you just ask it, it will invent something plausible. So instead of asking the model to *remember*,
you: **find** the relevant text yourself, **paste it into the prompt**, and ask the model to answer
*only from that*. Retrieval does the knowing; the LLM does the writing.

The whole pipeline in one line:

```
document -> chunks -> embeddings -> stored in Postgres
question -> embedding -> find nearest chunks -> paste into prompt -> LLM writes the answer
```

### Vocabulary you will meet in this notebook

| Term | What it actually means |
|---|---|
| **Chunk** | A document cut into a bite-sized piece (a few hundred words). You retrieve chunks, not whole documents, because pasting a 50-page PDF into every prompt is slow, expensive, and *makes answers worse* by burying the relevant sentence. |
| **Embedding** | A list of numbers (here, 1536 of them) representing a piece of text's *meaning*. Produced by an embedding model. Texts that mean similar things get similar numbers. |
| **Vector** | The embedding, once stored in the database. `vector(1536)` is a pgvector column type. |
| **Cosine distance** (`<=>`) | How far apart two embeddings are. **0 = identical meaning**, larger = less related. This is why we sort *ascending* — nearest first. |
| **pgvector** | The Postgres extension that adds the `vector` column type and the `<=>` operator. Without it, Postgres has no idea what a vector is. |
| **ANN / HNSW** | *Approximate Nearest Neighbour*. Comparing your question against all 10 million rows is slow, so HNSW builds a graph that finds the near-certain best matches by checking only a few hundred. "Approximate" = occasionally misses a borderline match, in exchange for being ~1000× faster. |
| **Full-text search / `tsvector`** | Classic **keyword** matching, built into Postgres. Knows nothing about meaning, but nails exact strings like `BID-2024-089`. |
| **Hybrid search** | Running the meaning-based search *and* the keyword search, then merging the two ranked lists. Each covers the other's blind spot. |
| **RRF** | *Reciprocal Rank Fusion* — the merging formula. It combines lists by **position** (1st, 2nd, 3rd) rather than by score, so the two searches' incompatible scoring scales never have to be reconciled. |
| **Reranker** | A second, smarter, slower pass. Stage 1 grabs 40 roughly-relevant candidates fast; the reranker reads them properly and keeps the best 4. |
| **`task_type`** | Tells the embedding model whether this text is a *document being stored* (`RETRIEVAL_DOCUMENT`) or a *question doing the searching* (`RETRIEVAL_QUERY`). The model deliberately encodes them differently. Getting this wrong silently costs you accuracy. |
| **recall@k** | Of your test questions, how often was a correct document among the top *k* results? "Did we find it at all?" |
| **MRR** | *Mean Reciprocal Rank* — how **high** the correct answer ranked. 1st place = 1.0, 2nd = 0.5, 3rd = 0.33. "Did we find it *first*?" |
| **nDCG** | Same idea as MRR but handles questions with several correct documents. |
| **Hallucination** | The model stating something confidently that the sources do not support. The prompt in Step 10 and the refusal rule exist entirely to prevent this. |

## Why ONE chunker and ONE model at serving time — and how you still justify that choice

There are **two separate activities** here, and conflating them is what made v1.1 confusing:

| | **Selection** (choosing) | **Serving** (running) |
|---|---|---|
| What it is | An experiment measuring which config wins | The product answering questions |
| How often | **Once**, offline, re-run when something changes | On **every user request** |
| How many pipelines | **As many as you can afford to test** | **Exactly one** — the winner |
| Lives in | **Step 13** | Steps 2–12 |

So: *yes, absolutely test many chunkers and many models* — in Step 13. Then **deploy one**. Serving 27
pipelines is not a thing anyone does; it would multiply your storage, cost and latency by 27 to produce
27 different answers to the same question with no way to choose between them at request time.

### "Why didn't you use different chunking methods?"

The right answer is **"I did — here are the numbers."** That is what Step 13 produces: a table
comparing v1.1's three chunkers *plus* the recursive splitter, on your data, with your questions.
Run it, keep the output, put it in the deck. That answer is far stronger than either
*"I tested 27 combos"* (then why is one deployed?) or *"I picked a sensible default"* (on what basis?).

### "Why gemini-embedding-001 and not gecko@003 / 004 / 005?"

First, get the lineage right so you sound informed — these are **generations, not options**:

```
textembedding-gecko@001/@002/@003   ->  text-embedding-004/005  ->  gemini-embedding-001
        (legacy, 768d)                      (768d)                   (3072d, current)
```

They are successive generations of the *same* product line. Picking `gecko@003` over
`gemini-embedding-001` is like specifying an older model year — you would need a *reason*, such as an
existing index you cannot afford to re-embed. Check the **Vertex AI model lifecycle page** for current
deprecation and retirement dates before you quote any of them to a client; the older ones are on the
way out, and "it gets retired in N months" is itself a decisive argument.

Second, give three reasons in this order — **only the first is really about your data**:

1. **Measured** — it won your own benchmark. v1.1's model comparison was sound: `gemini-embedding-001`
   scored MRR **0.90** vs **0.70** for `text-embedding-004`/`005`. (v1.1's *index* comparison was the
   part that measured noise — see the intro. Do not quote that half.)
2. **Published benchmarks** — it ranks at/near the top of **MTEB**, the standard public embedding
   leaderboard. Useful supporting evidence, never a substitute for testing on your own data: MTEB is
   general web/academic text, not procurement documents.
3. **Operational** — current generation (longest support runway), multilingual, and **Matryoshka
   (MRL)**, which lets you truncate 3072 → 1536 → 768 dims and trade accuracy for storage *without
   re-embedding*. That flexibility is a genuine architectural advantage the older models do not have.

### ⚠️ The question that will actually catch you out

> *"How many questions did you evaluate on?"*

You said you tested on **2–3 documents**. Be aware of what that can and cannot support:

- With **13 questions**, one question flipping changes recall by **~8 percentage points**. With 5, it is
  **20 points**. The real difference between two decent configs is often 2–5 points — **smaller than
  your measurement noise.**
- That is exactly why v1.1's leaderboard was full of ties (`0.80`, `0.80`, `0.80`...). It was not
  measuring that the configs were equal; it lacked the power to tell them apart.
- **Rule of thumb: ~50 questions minimum to rank configurations, 100+ to trust a small gap.** They do
  not need to be hard to write — pull them from real user logs, or have two colleagues write 25 each
  from documents they know.

**What to say if you are asked before you have that:** *"Current results are directional, from a
{N}-question pilot set on {M} documents. I'm expanding to 50+ questions from real queries before we
lock the configuration."* That is a professional answer. Presenting 5-question results as conclusive is
the thing that damages credibility.

### The one hard constraint behind all of this

> **Vectors from different embedding models are not comparable.** `text-embedding-004` and
> `gemini-embedding-001` place "romaine lettuce" at completely different coordinates. Comparing one to
> the other returns a meaningless number — not an error, just silent garbage.

Which is why:

- **One table = one model, permanently.** That is why v1.1 built a *separate table per model* — it had
  no choice, and why Step 13 does the same for its throwaway tables.
- **Changing `EMBED_MODEL` or `EMBED_DIM` invalidates every stored vector.** You must re-embed the whole
  corpus (`ingest(force=True)`). It is a migration, not a config tweak — which is precisely why you run
  Step 13 *before* you commit, not after.

## Step 1 — Install

In [ ]:
# google-genai     -> Vertex AI (embeddings + Gemini)
# psycopg2-binary  -> PostgreSQL / Cloud SQL
# tiktoken         -> token counting for chunk budgeting
%pip install -q google-genai "psycopg2-binary>=2.9" tiktoken
print("Installed. Restart the kernel if pip asked you to.")

## Step 2 — Configuration

Credentials come from the environment, not from this file. Set them once in your shell, `.env`, or
Secret Manager:

```bash
export PGHOST=10.151.179.4 PGDATABASE=postgres PGUSER=postgres PGPASSWORD='...'
export GCP_PROJECT_ID=div-aais-rfpiq-usc1-uat GCP_LOCATION=us-central1
```

**On `EMBED_DIM = 1536`:** `gemini-embedding-001` emits 3072 dims natively, which exceeds pgvector's
**2000-dim ceiling for `hnsw`/`ivfflat`** — that is the sole reason v1.1's leaderboard picked "no index".
The model is Matryoshka-trained, so truncating to 1536 (and re-normalising, which Step 5 does) retains
essentially all retrieval quality *and* halves storage. That buys HNSW back. You can go to 768 to halve
it again — benchmark with Step 11 before you do.

In [ ]:
import os   # gives us os.environ, the dictionary of environment variables

# =========================== Vertex AI (Google Cloud) ===========================
# os.environ.get("NAME", fallback) reads an environment variable, using `fallback`
# if it is not set. That is what lets you override any of these without editing code.
PROJECT_ID   = os.environ.get("GCP_PROJECT_ID", "div-aais-rfpiq-usc1-uat")  # which GCP project bills the API calls
LOCATION     = os.environ.get("GCP_LOCATION", "us-central1")                # which region serves the models

EMBED_MODEL  = "gemini-embedding-001"  # the model that turns text -> numbers. ONE model, forever:
                                       # vectors from different models are NOT comparable (see the
                                       # "Why v2 has ONE chunker" section above).
EMBED_DIM    = 1536                    # how many numbers per embedding. This model emits 3072 natively,
                                       # but pgvector's hnsw index refuses anything over 2000 dims.
                                       # The model is "Matryoshka"-trained: you can cut it short and it
                                       # still works. 1536 = full quality, half the storage, index allowed.
CHAT_MODEL   = "gemini-2.5-flash"      # writes the final answer (Step 10)
RERANK_MODEL = "gemini-2.5-flash"      # scores search results (Step 9); swap for Vertex Ranking API in prod

# =========================== Postgres / Cloud SQL ===============================
# NEVER hard-code a password in a notebook - v1.1 did, and that credential is now
# in every copy of that file. Read it from the environment instead.
_pw = os.environ.get("PGPASSWORD")
if not _pw:                              # not set? ask interactively rather than crashing
    import getpass                       # getpass hides what you type (unlike input())
    _pw = getpass.getpass("Postgres password: ")

DB_CONFIG = {
    "host":     os.environ.get("PGHOST", "10.151.179.4"),  # Cloud SQL private IP, or "localhost" via Auth Proxy
    "port":     int(os.environ.get("PGPORT", 5432)),       # int() because env vars are always strings
    "dbname":   os.environ.get("PGDATABASE", "postgres"),
    "user":     os.environ.get("PGUSER", "postgres"),
    "password": _pw,
    "connect_timeout": 10,        # give up after 10s instead of hanging forever on a bad network route
    "application_name": "rag_v2", # shows up in pg_stat_activity - lets you spot YOUR queries on the server
}

DOC_TABLE   = "rag_documents"   # raw, un-chunked source text
CHUNK_TABLE = "rag_chunks"      # the chunks + their vectors (what search actually reads)

# =========================== Chunking (Step 6) ==================================
CHUNK_TOKENS         = 350   # max size of one chunk, in tokens (~260 words / ~1400 chars).
                             # Too big -> the relevant sentence gets buried and retrieval blurs.
                             # Too small -> facts get split apart and lose their context.
                             # 250-500 is the practical sweet spot for question-answering over prose.
CHUNK_OVERLAP_TOKENS = 60    # ~15% of each chunk is repeated from the previous one, so a fact sitting
                             # exactly on a boundary still appears whole in at least one chunk.
MAX_EMBED_TOKENS     = 2000  # safety cap per embedding call (gemini-embedding-001 accepts 2048).

# =========================== Retrieval (Steps 8-9) ==============================
# Retrieval runs in two stages: cast a WIDE net cheaply, then narrow it carefully.
VECTOR_CANDIDATES   = 40    # stage 1a: how many chunks the meaning-based search returns
TEXT_CANDIDATES     = 40    # stage 1b: how many chunks the keyword search returns
RRF_K               = 60    # the constant in the RRF formula, weight = 1/(k + rank). Bigger k = flatter
                            # weighting (rank 1 and rank 5 treated more equally). 60 is the published default.
W_VECTOR, W_TEXT    = 1.0, 1.0   # relative trust in each search. Raise W_TEXT if your users search by
                                 # exact IDs/SKUs; raise W_VECTOR if they ask conceptual questions.
TOP_K               = 8     # how many survive the merge when we are NOT reranking
RERANK_CANDIDATES   = 20    # how many survive the merge and get handed to the reranker
RERANK_TOP_N        = 4     # stage 2: how many the LLM finally sees. Small on purpose - irrelevant
                            # passages in the prompt measurably make answers worse.
USE_RERANKER        = True
MAX_COSINE_DISTANCE = 0.80  # relevance floor. If the closest chunk is further than this AND no keyword
                            # matched, we refuse rather than answer from unrelated documents.

# =========================== HNSW index tuning (Step 7c) ========================
HNSW_M                = 16   # graph connections per node. Higher = better recall, bigger index, slower build.
HNSW_EF_CONSTRUCTION  = 64   # how hard the index works while BUILDING. Higher = better index, slower build.
HNSW_EF_SEARCH        = 100  # how hard it works while SEARCHING. This is the one you actually tune -
                             # it is free to change (no rebuild) and trades latency directly for recall.

print(f"Config loaded. {EMBED_MODEL} @ {EMBED_DIM}d -> {DB_CONFIG['host']}/{DB_CONFIG['dbname']}")

## Step 3 — Connection pool

v1.1 opened a brand-new TCP + TLS + auth handshake for *every single query*. Under load that alone
costs tens of milliseconds per call and will exhaust Cloud SQL's connection limit. Production pools.

In [ ]:
import contextlib        # provides @contextmanager, which lets us write our own `with` blocks
import psycopg2          # the PostgreSQL driver for Python
import psycopg2.extras   # extra cursor types - RealDictCursor returns dicts instead of tuples
from psycopg2.pool import ThreadedConnectionPool

# A connection pool opens a few connections up front and lends them out on demand,
# instead of doing a fresh TCP + TLS + password handshake for every query.
#   minconn=1 -> always keep at least one connection alive and ready
#   maxconn=8 -> never open more than 8 (the Cloud SQL server has a global limit;
#                if you run 4 app instances at maxconn=8 each, that is 32 connections)
#   **DB_CONFIG -> unpacks the dict from Step 2 into host=..., port=..., etc.
POOL = ThreadedConnectionPool(minconn=1, maxconn=8, **DB_CONFIG)

@contextlib.contextmanager   # turns the function below into something usable as `with db() as cur:`
def db(dict_rows: bool = False):
    """Borrow a pooled connection and hand back a cursor.

    Everything before `yield` runs on entering the `with` block; everything after it runs
    on exit - including when an exception was raised. That is what guarantees cleanup.

    dict_rows=True -> rows come back as dicts (row["doc_id"]); False -> tuples (row[0]).
    """
    conn = POOL.getconn()          # borrow a connection from the pool
    try:
        # Choose the cursor type. `None` means psycopg2's default tuple-returning cursor.
        factory = psycopg2.extras.RealDictCursor if dict_rows else None

        # `with conn:` is psycopg2's TRANSACTION block, not a close-the-connection block.
        # Leaving it normally  -> COMMIT.  Leaving it via an exception -> ROLLBACK.
        # So a failed multi-step write can never leave the database half-updated.
        with conn:
            # `with conn.cursor(...)` closes the cursor and frees its server-side resources.
            with conn.cursor(cursor_factory=factory) as cur:
                yield cur          # hand the cursor to the caller's `with` block
    finally:
        # `finally` runs no matter what - success, exception, even an early return.
        # Without this line a crash would permanently lose a connection from the pool
        # ("connection leak"), and after 8 crashes the app would hang forever.
        POOL.putconn(conn)

# Smoke test: prove we can reach the database before any of the expensive steps run.
with db() as cur:
    cur.execute("SELECT current_database(), version()")
    dbname, version = cur.fetchone()   # fetchone() -> the first row, here a 2-item tuple
print(f"Connected to '{dbname}' - {version.split(',')[0]}")

## Step 4 — Schema

Two tables, not one-table-per-experiment:

- **`rag_documents`** — raw source text + JSONB metadata + `content_hash` (what the document *is* now)
  and `indexed_hash` (what we last embedded). Comparing the two is how Step 7c re-embeds only what changed.
- **`rag_chunks`** — one row per chunk:
  - `content` — clean text shown to the LLM and the user
  - `embed_input` — `content` **plus its contextual header**; this is what got embedded and what
    full-text search runs over, so a chunk stays findable by its supplier name even when the chunk body
    never mentions it. **This is the single change that fixes your satisfaction question.**
  - `metadata jsonb` — denormalised from the parent doc so you can filter *before* the ANN search
  - `tsv` — a **generated** tsvector column: Postgres maintains it, so it can never drift from the text
  - `ON DELETE CASCADE` — deleting a document cannot leave orphaned vectors behind

The vector index is built in Step 7c *after* the bulk load. Creating HNSW on an empty table and then
inserting row-by-row is markedly slower than loading first and indexing once.

In [ ]:
# An f-string (f""" ... """) substitutes {DOC_TABLE}, {EMBED_DIM} etc. with the Step 2 values.
# Because of that, any brace we want to keep LITERAL in the SQL must be doubled: '{{}}' -> '{}'.
DDL = f"""
-- pgvector adds the `vector` column type and the <=> distance operator. Without it the
-- CREATE TABLE below fails with 'type "vector" does not exist'. Safe to re-run.
CREATE EXTENSION IF NOT EXISTS vector;

-- ============ Table 1: the original documents, exactly as you supplied them ============
CREATE TABLE IF NOT EXISTS {DOC_TABLE} (
    doc_id        text PRIMARY KEY,     -- your own stable id, e.g. 'acme-produce-2024'
    title         text NOT NULL,
    content       text NOT NULL,        -- the full, un-chunked text
    metadata      jsonb NOT NULL DEFAULT '{{}}'::jsonb,  -- supplier, category, dates... free-form.
                                        -- jsonb (not json) is the binary form: indexable and faster.
                                        -- Using one jsonb column instead of 10 fixed columns means
                                        -- adding a new field later needs no schema migration.
    content_hash  text NOT NULL,        -- fingerprint of what this document says RIGHT NOW
    indexed_hash  text,                 -- fingerprint of what we last EMBEDDED. NULL = never indexed.
                                        -- content_hash != indexed_hash  =>  needs re-embedding.
                                        -- Comparing these two is the whole trick behind Step 7c.
    created_at    timestamptz NOT NULL DEFAULT now(),
    updated_at    timestamptz NOT NULL DEFAULT now()
);

-- ============ Table 2: the chunks + their vectors (what search actually reads) ==========
CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
    chunk_id     bigserial PRIMARY KEY,  -- bigserial = auto-incrementing 64-bit id
    doc_id       text NOT NULL REFERENCES {DOC_TABLE}(doc_id) ON DELETE CASCADE,
                                         -- REFERENCES = this must match a real document.
                                         -- ON DELETE CASCADE = deleting a document automatically
                                         -- deletes its chunks, so you can never leave orphaned
                                         -- vectors behind that still show up in search results.
    chunk_index  int  NOT NULL,          -- 0, 1, 2... position of this chunk within its document
    content      text NOT NULL,          -- CLEAN text: what we show the LLM and the user
    embed_input  text NOT NULL,          -- content + its contextual header: what we actually EMBEDDED
                                         -- and what keyword search reads. Keeping the two separate is
                                         -- what fixes the satisfaction bug without showing the user
                                         -- an ugly metadata header. See Step 6.
    token_count  int  NOT NULL,          -- stored so you can audit chunk sizes without re-tokenising
    metadata     jsonb NOT NULL DEFAULT '{{}}'::jsonb,  -- copied down from the parent document so we
                                         -- can filter DURING the vector search, not after it
    embedding    vector({EMBED_DIM}) NOT NULL,  -- the {EMBED_DIM} numbers. The size is fixed at table
                                         -- creation: another reason one table = one embedding model.

    -- A GENERATED column is computed by Postgres and kept up to date automatically - you never
    -- write to it, and it can never drift out of sync with the text. to_tsvector() strips
    -- punctuation, lowercases, and reduces words to stems ('deliveries' -> 'deliveri') so that
    -- searching "delivery" also matches "deliveries". coalesce(x,'') guards against NULL.
    tsv tsvector GENERATED ALWAYS AS (to_tsvector('english', coalesce(embed_input, ''))) STORED,

    created_at   timestamptz NOT NULL DEFAULT now(),
    UNIQUE (doc_id, chunk_index)         -- one row per (document, position). Makes re-ingest safe.
);

-- ============ Indexes: how Postgres avoids scanning every row =========================
-- GIN ("generalised inverted index") is the right index type for search-inside-a-value
-- columns. It maps each word -> the rows containing it, like the index at the back of a book.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_tsv_gin  ON {CHUNK_TABLE} USING gin (tsv);
-- jsonb_path_ops is a smaller, faster GIN variant that supports the @> ("contains") operator,
-- which is exactly what our metadata filters use.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_meta_gin ON {CHUNK_TABLE} USING gin (metadata jsonb_path_ops);
-- Plain B-tree index: makes "DELETE ... WHERE doc_id = ..." during re-ingest fast.
CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_doc_id   ON {CHUNK_TABLE} (doc_id);
"""
# NOTE: the HNSW *vector* index is deliberately NOT created here - see Step 7c. Building it
# on an empty table and then inserting is much slower than loading first, indexing once.

with db() as cur:
    cur.execute(DDL)   # psycopg2 happily runs several statements separated by ';' in one call
print(f"Schema ready: {DOC_TABLE}, {CHUNK_TABLE} (vector({EMBED_DIM}) + tsvector + jsonb).")

## Step 5 — Embeddings: batched, retried, normalised

Four things v1.1's `embed_texts` lacked:

- **Retry with exponential backoff + jitter.** Vertex returns `429 RESOURCE_EXHAUSTED` under load.
  v1.1's bare `except` retried once, one text at a time, then let the exception kill the run.
- **L2 normalisation.** Once you truncate an MRL embedding below its native 3072 dims the vector is no
  longer unit length. Cosine still works, but normalising keeps `<=>` numerically well behaved and makes
  distances comparable across models.
- **Input truncation.** One oversized chunk otherwise 400s the entire batch.
- **A query-embedding cache.** Re-embedding the same question is ~200 ms of the request budget wasted.

In [ ]:
import math, random, time
from functools import lru_cache

import tiktoken
from google import genai
from google.genai.types import EmbedContentConfig

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# cl100k_base is only a consistent yardstick for budgeting chunk sizes - it does not need
# to match the embedding model's own tokenizer.
_enc = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    """How many tokens is this text? Used to keep chunks inside CHUNK_TOKENS."""
    return len(_enc.encode(text))

def _truncate(text: str, max_tokens: int = MAX_EMBED_TOKENS) -> str:
    """Cut text down to max_tokens. One oversized input would otherwise fail the whole batch."""
    toks = _enc.encode(text)                                  # text -> list of token ids
    return text if len(toks) <= max_tokens else _enc.decode(toks[:max_tokens])   # and back again

def _l2_normalize(vec):
    """Scale a vector so its length is exactly 1.0, without changing its direction.

    Why: cosine distance only cares about direction (the angle), not length. Truncating a
    3072-dim embedding to 1536 leaves it slightly shorter than 1.0, so we re-normalise to
    keep <=> numerically well behaved and distances comparable between models.
    """
    norm = math.sqrt(sum(x * x for x in vec))    # Pythagoras in 1536 dimensions
    return [x / norm for x in vec] if norm else vec   # `if norm` guards against divide-by-zero

# Error signatures that mean "the server is busy, try again" rather than "you did it wrong".
_TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL", "DEADLINE")

def _with_retry(fn, attempts: int = 5, base: float = 1.0):
    """Call fn(); if it fails with a TEMPORARY error, wait and try again.

    `fn` is a function, not a value - callers pass `lambda: something()` so we can
    re-invoke it. Waits 1s, 2s, 4s, 8s ("exponential backoff"), plus a random fraction
    of a second ("jitter") so that many parallel workers do not all retry in lockstep
    and hammer the server at the same instant.
    """
    for i in range(attempts):
        try:
            return fn()                       # success - return immediately
        except Exception as e:
            # Give up if we are out of attempts, OR if this error will never fix itself.
            # A 403 (no permission) or a typo in the model name is permanent: retrying it
            # 5 times just wastes 15 seconds and hides the real problem.
            if i == attempts - 1 or not any(s in str(e) for s in _TRANSIENT):
                raise
            time.sleep(base * (2 ** i) + random.random())   # 2**i -> 1, 2, 4, 8...

def embed_texts(texts, task_type: str, model: str = EMBED_MODEL,
                dim: int = EMBED_DIM, batch_size: int = 16):
    """Turn a list of strings into a list of vectors.

    task_type must be RETRIEVAL_DOCUMENT for stored chunks and RETRIEVAL_QUERY for
    questions. The model deliberately encodes the two roles differently ("what can be
    found" vs "what is doing the finding"), and mixing them up silently costs accuracy -
    no error, just worse results.
    """
    cfg = EmbedContentConfig(task_type=task_type, output_dimensionality=dim)
    out = []
    # Send 16 texts per API call instead of one call per text: far fewer network round trips.
    # range(0, len, step) walks the list in blocks: 0..15, 16..31, ...
    for i in range(0, len(texts), batch_size):
        batch = [_truncate(t) for t in texts[i:i + batch_size]]
        try:
            resp = _with_retry(lambda: genai_client.models.embed_content(
                model=model, contents=batch, config=cfg))
            out.extend(e.values for e in resp.embeddings)   # .values is the list of floats
        except Exception:
            # Fallback: some model/quota combinations reject batches and accept only one
            # input per call. Retry this block one text at a time rather than failing.
            for t in batch:
                # `lambda t=t:` captures the CURRENT t. Without the `t=t` every lambda would
                # share the loop variable and embed the last text repeatedly - a classic bug.
                resp = _with_retry(lambda t=t: genai_client.models.embed_content(
                    model=model, contents=[t], config=cfg))
                out.append(resp.embeddings[0].values)
    return [_l2_normalize(v) for v in out]

def embed_query(question: str):
    """Embed ONE question. [0] because embed_texts always returns a list."""
    return embed_texts([question], "RETRIEVAL_QUERY")[0]

@lru_cache(maxsize=1024)   # remembers the last 1024 questions: same question -> instant, no API call
def embed_query_cached(question: str):
    """Production path. Saves ~200ms on any repeated question.

    Returns a tuple, not a list, because @lru_cache requires everything it stores to be
    hashable (immutable) - and lists are not. Callers convert back with list(...).

    Step 11's benchmark deliberately calls the UNCACHED embed_query() instead, so its
    latency numbers reflect a real cold request rather than a cache hit.
    """
    return tuple(embed_query(question))

_probe = embed_query("hello")
assert len(_probe) == EMBED_DIM, f"expected {EMBED_DIM} dims, got {len(_probe)}"
print(f"Vertex OK - {EMBED_MODEL} -> {len(_probe)} dims, "
      f"|v| = {math.sqrt(sum(x * x for x in _probe)):.4f}")

## Step 6 — Structure-aware chunking + contextual headers

**Chunking.** v1.1's three chunkers were all naive: `fixed_400` slices mid-word, `token_256` slices
mid-sentence. Production uses a *recursive* strategy — respect the largest natural boundary that fits,
and fall back to a harder cut only when you must:

> paragraph → sentence → hard token split, then pack greedily to a token budget with sentence-aligned overlap.

**Contextual headers — the actual fix for your bug.** Before embedding, each chunk is prefixed with its
document identity:

```
Document: Q1 2024 Supplier Performance Review | ID: performance-review-2024 |
Supplier Name: Riverside USD | Category: Performance Review | Bid Date: 2024-04-01

Q1 2024 Performance Review Summary: Acme Foods maintained 98% on-time delivery ...
```

A chunk is now retrievable by *"performance review"*, *"Riverside"* or *"Q1 2024"* even when the chunk
body never repeats those words, and the LLM can see which document each fact came from. Anthropic's
published *contextual retrieval* work measures roughly a 35% reduction in retrieval failures from this
idea alone. (Their full version generates a per-chunk LLM summary — the obvious upgrade when budget
allows; the header form below is free.)

`content` (clean) and `embed_input` (header + clean) are stored separately, so the header improves
retrieval without polluting the text you show the user.

> **Note on this demo corpus.** All six sample documents are 70–140 tokens, so at `CHUNK_TOKENS = 350`
> each becomes exactly **one** chunk and the splitter never actually fires. That is deliberate and worth
> understanding: on *this* data the satisfaction bug is fixed by the header, hybrid retrieval and the
> prompt rules — not by lucky chunk boundaries. The splitter matters once you point Step 7b at real
> multi-page RFPs. Re-run Step 11 after you do, because that is when `CHUNK_TOKENS` starts to matter.

In [ ]:
import re   # regular expressions: pattern-matching on text

# A blank line = a paragraph break. \n is a newline, \s* is "any whitespace, possibly none".
# So this matches "newline, optional blank space, newline".
_PARA = re.compile(r"\n\s*\n")

# A sentence boundary. Read it in three parts:
#   (?<=[.!?])      "lookbehind": the character BEFORE this point must be . ! or ?
#   \s+             one or more spaces/newlines (this is the part actually removed)
#   (?=[A-Z0-9("'])  "lookahead": what FOLLOWS must start like a new sentence
# The lookahead is what stops it splitting "3.20 USD" or "Net-30. delivery" incorrectly.
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\"'])")

def _hard_split(unit: str, max_tokens: int):
    """Last resort: chop blindly at token boundaries.

    Only reached when a single 'sentence' is longer than the ENTIRE chunk budget - e.g. a
    giant table row or a wall of text with no punctuation. Ugly, but better than emitting a
    chunk too big to embed.
    """
    toks = _enc.encode(unit)
    return [_enc.decode(toks[i:i + max_tokens]) for i in range(0, len(toks), max_tokens)]

def split_text(text: str, max_tokens: int = CHUNK_TOKENS,
               overlap_tokens: int = CHUNK_OVERLAP_TOKENS):
    """Cut one document into chunks. This is THE chunker - v2 has exactly one.

    Two phases:
      Phase 1 - break the text into the smallest sensible units (sentences).
      Phase 2 - glue those units back together into chunks just under the token budget.

    This is called "recursive" splitting: try the biggest natural boundary first
    (paragraph), fall back to a smaller one (sentence), and only cut mid-sentence when
    there is genuinely no alternative.
    """
    # ---------- Phase 1: text -> list of sentences ----------
    units = []
    for para in _PARA.split(text.strip()):      # first split on blank lines
        para = para.strip()
        if not para:                            # skip empty results
            continue
        for sent in _SENT.split(para):          # then split each paragraph into sentences
            sent = sent.strip()
            if not sent:
                continue
            # If even one sentence blows the budget, chop it; otherwise keep it whole.
            units.extend(_hard_split(sent, max_tokens) if n_tokens(sent) > max_tokens else [sent])

    # ---------- Phase 2: sentences -> chunks ----------
    chunks = []      # finished chunks
    buf = []         # sentences accumulated for the chunk currently being built
    buf_tokens = 0   # running token count of buf, so we do not re-count it every iteration

    for unit in units:
        t = n_tokens(unit)

        # Would adding this sentence overflow the budget? Then finish the current chunk first.
        # `buf and` prevents an empty chunk on the very first iteration.
        if buf and buf_tokens + t > max_tokens:
            chunks.append(" ".join(buf))        # emit the completed chunk

            # ----- overlap: start the NEXT chunk with the last few sentences of this one -----
            # Why: if a fact sits exactly on a chunk boundary ("...satisfaction was" | "4.8/5
            # stars"), neither chunk contains it in full and retrieval loses it. Repeating a
            # little text means the fact appears intact in at least one chunk.
            # We walk BACKWARDS from the end, taking whole sentences until we hit the overlap
            # budget - whole sentences only, so overlap never starts mid-thought.
            keep, kept = [], 0
            for prev in reversed(buf):
                pt = n_tokens(prev)
                if kept + pt > overlap_tokens:  # one more would exceed the overlap budget
                    break
                keep.insert(0, prev)            # insert(0,...) rebuilds the original order
                kept += pt
            buf, buf_tokens = keep, kept        # the new chunk starts with the carried-over text

        buf.append(unit)
        buf_tokens += t

    if buf:                       # whatever is left over becomes the final chunk
        chunks.append(" ".join(buf))
    return chunks

# ======================= Contextual headers: the fix for your bug =======================
# Which metadata fields are worth spending header tokens on - the ones users search BY.
# Adding every field would dilute the embedding; these are the high-value ones.
HEADER_FIELDS = ["supplier_name", "category", "document_type", "bid_date", "status"]

def contextual_header(doc_id: str, title: str, metadata: dict) -> str:
    """Build a one-line 'where did this text come from' banner.

    Produces e.g.:
      Document: Q1 2024 Supplier Performance Review | ID: performance-review-2024 |
      Supplier Name: Riverside USD | Category: Performance Review | Bid Date: 2024-04-01
    """
    bits = [f"Document: {title}", f"ID: {doc_id}"]
    # For each interesting field that is actually present, add "Field Name: value".
    # .replace('_',' ').title() turns 'supplier_name' into 'Supplier Name'.
    # `if metadata.get(f)` skips fields that are missing, None, or empty.
    bits += [f"{f.replace('_', ' ').title()}: {metadata[f]}"
             for f in HEADER_FIELDS if metadata.get(f)]
    return " | ".join(bits)

def build_chunks(doc_id: str, title: str, content: str, metadata: dict):
    """Split a document AND attach its context header to every piece.

    Returns a list of dicts, one per chunk. The key idea is that 'content' and
    'embed_input' differ:
      content     -> clean text, shown to the LLM and the user
      embed_input -> header + clean text, what we embed and keyword-index
    So the header makes the chunk findable without ever appearing in the answer.
    """
    header = contextual_header(doc_id, title, metadata)   # same header for every chunk of this doc
    return [
        {
            "chunk_index": i,                       # enumerate() gives us 0, 1, 2, ...
            "content": piece,                       # shown to the LLM / user
            "embed_input": f"{header}\n\n{piece}",  # embedded + full-text indexed
            "token_count": n_tokens(piece),
        }
        for i, piece in enumerate(split_text(content))
    ]

_demo = build_chunks(
    "performance-review-2024", "Q1 2024 Supplier Performance Review",
    "Acme Foods maintained 98% on-time delivery. Customer satisfaction: 4.8/5 stars.",
    {"supplier_name": "Riverside USD", "category": "Performance Review", "bid_date": "2024-04-01"},
)
print(_demo[0]["embed_input"])

## Step 7a — Load source documents

The same six documents as v1.1, but metadata now lands in one JSONB column instead of ten fixed
columns. JSONB means adding a facet (`region`, `contract_id`, `confidentiality`) never needs a
migration, and `metadata @> '{"category":"Dairy"}'` is served by the GIN index from Step 4.

`content_hash` covers title + content + metadata, so *any* edit — including a metadata-only edit that
changes the contextual header — correctly marks the document for re-embedding.

In [ ]:
import hashlib, json
from psycopg2.extras import execute_values

def content_hash(title: str, content: str, metadata: dict) -> str:
    payload = json.dumps({"t": title, "c": content, "m": metadata},
                         sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def upsert_documents(doc_map: dict):
    """doc_map: {doc_id: {"title": str, "content": str, "metadata": dict}}.
    Idempotent - re-running with unchanged text is a no-op that Step 7c will skip."""
    rows = []
    for doc_id, d in doc_map.items():
        title, content = d["title"], d["content"]
        meta = d.get("metadata", {})
        rows.append((doc_id, title, content, json.dumps(meta), content_hash(title, content, meta)))

    with db() as cur:
        execute_values(cur, f"""
            INSERT INTO {DOC_TABLE} (doc_id, title, content, metadata, content_hash) VALUES %s
            ON CONFLICT (doc_id) DO UPDATE SET
                title        = EXCLUDED.title,
                content      = EXCLUDED.content,
                metadata     = EXCLUDED.metadata,
                content_hash = EXCLUDED.content_hash,
                updated_at   = now()
        """, rows, template="(%s, %s, %s, %s::jsonb, %s)")
    print(f"Upserted {len(rows)} document(s).")

def delete_documents(doc_ids):
    """FK cascade removes the chunks and their vectors - no orphans."""
    doc_ids = list(doc_ids)
    with db() as cur:
        cur.execute(f"DELETE FROM {DOC_TABLE} WHERE doc_id = ANY(%s)", (doc_ids,))
    print(f"Deleted {len(doc_ids)} document(s) and their chunks.")

print("upsert_documents() / delete_documents() ready.")

In [ ]:
SOURCE_DOCS = {
    "acme-produce-2024": {
        "title": "Acme Foods - Produce Quote",
        "content": (
            "Acme Foods bid on the produce category for Riverside USD on 2024-05-01. "
            "Romaine lettuce: 3.20 USD/case. Spinach: 4.10 USD/case. Carrots: 2.05 USD/case. "
            "Brussels sprouts: 2.85 USD/case. Bell peppers (mixed): 3.50 USD/case. "
            "Payment terms: Net-30. Delivery: twice weekly on Tuesdays and Thursdays. "
            "Minimum order: $100. Volume discount: 5% on orders over $500."
        ),
        "metadata": {
            "supplier_name": "Acme Foods Inc.", "category": "Produce",
            "bid_date": "2024-05-01", "document_type": "Supplier Quote",
            "status": "Active", "delivery_terms": "Twice weekly (Tue/Thu)",
            "price_range": "$2.05-$4.10 per case", "bid_id": "BID-2024-089",
        },
    },
    "globex-dairy-2024": {
        "title": "Globex Dairy - Dairy Products Quote",
        "content": (
            "Globex Dairy responded to the dairy RFP for Riverside USD on 2024-05-02. "
            "Whole milk (gallon): 1.85 USD/gallon. 2% milk (gallon): 1.75 USD/gallon. "
            "Skim milk (gallon): 1.65 USD/gallon. Cheddar block (lb): 3.40 USD/lb. "
            "Mozzarella (lb): 2.95 USD/lb. Butter (lb): 4.20 USD/lb. "
            "Payment terms: Net-45. Early payment discount: 2% if paid within 10 days. "
            "They requested a 12-month contract with quarterly price reviews. "
            "Delivery: Monday-Friday with 24-hour notice. Cold chain guaranteed."
        ),
        "metadata": {
            "supplier_name": "Globex Dairy Inc.", "category": "Dairy",
            "bid_date": "2024-05-02", "document_type": "Supplier Quote",
            "status": "Active", "delivery_terms": "Mon-Fri (24-hour notice)",
            "price_range": "$1.65-$4.20 per unit", "bid_id": "BID-2024-089",
        },
    },
    "bid-summary-2024": {
        "title": "Riverside USD RFP Summary - BID-2024-089",
        "content": (
            "Bid BID-2024-089 for Riverside USD School District: Comprehensive food service RFP. "
            "Eight suppliers solicited across five categories: Produce, Dairy, Proteins, "
            "Beverages, and Pantry. Response rate: five responded (62.5%). "
            "Internal due date: 2024-05-20. Customer due date: 2024-05-25. "
            "Bid opening: 2024-06-01 at 2:00 PM. "
            "Status: Active and under review. Budget allocation: $500,000 annually. "
            "Contract period: July 1, 2024 - June 30, 2025 with renewal options. "
            "Evaluation criteria: price (40%), quality (35%), delivery reliability (15%), service (10%)."
        ),
        "metadata": {
            "supplier_name": "Riverside USD", "category": "RFP Summary",
            "bid_date": "2024-05-01", "document_type": "RFP", "status": "Active",
            "delivery_terms": "Multi-category", "price_range": "$500K annual budget",
            "bid_id": "BID-2024-089",
        },
    },
    "sysco-proteins-2024": {
        "title": "Sysco - Proteins & Meats Quote",
        "content": (
            "Sysco Foodservice quoted for protein category serving Riverside USD on 2024-05-03. "
            "Chicken breast (boneless, skinless, lb): 4.50 USD/lb. "
            "Ground beef 80/20 (lb): 5.20 USD/lb. "
            "Ground beef 90/10 (lb): 5.85 USD/lb. "
            "Salmon fillets (wild-caught, lb): 12.80 USD/lb. "
            "Tilapia fillets (lb): 6.40 USD/lb. "
            "Turkey breast (sliced, lb): 4.95 USD/lb. "
            "Volume discount: 5% on orders over 500 lbs. "
            "Payment terms: Net-60 with 1% early payment discount if paid by 10th. "
            "Delivery: Monday-Friday with 48-hour notice preferred. "
            "Cold chain maintained. USDA inspection certified. Halal options available."
        ),
        "metadata": {
            "supplier_name": "Sysco Foodservice", "category": "Proteins",
            "bid_date": "2024-05-03", "document_type": "Supplier Quote",
            "status": "Active", "delivery_terms": "Mon-Fri (48-hour notice)",
            "price_range": "$4.50-$12.80 per lb", "bid_id": "BID-2024-089",
        },
    },
    "us-foods-beverages-2024": {
        "title": "US Foods - Beverages & Juices Quote",
        "content": (
            "US Foods provided comprehensive beverages quote for Riverside USD on 2024-05-04. "
            "Orange juice concentrate (gallon): 2.30 USD/gallon. "
            "Apple juice (gallon): 1.95 USD/gallon. "
            "Cranberry juice (gallon): 2.75 USD/gallon. "
            "Milk 2% (gallon): 3.15 USD/gallon. "
            "Chocolate milk (gallon): 3.45 USD/gallon. "
            "Bottled water (case/24 bottles): 4.20 USD/case. "
            "Coffee (ground, 2lb bag): 8.50 USD/bag. "
            "Tea assortment (box/50 bags): 6.75 USD/box. "
            "Minimum order: $250. Free delivery for orders over $500. "
            "Payment: Net-45 with 2% early payment discount (10 days). "
            "Special: 10% discount on annual contracts signed by June 30, 2024."
        ),
        "metadata": {
            "supplier_name": "US Foods Inc.", "category": "Beverages",
            "bid_date": "2024-05-04", "document_type": "Supplier Quote",
            "status": "Active", "delivery_terms": "Flexible (free delivery >$500)",
            "price_range": "$1.95-$8.50 per unit", "bid_id": "BID-2024-089",
        },
    },
    "performance-review-2024": {
        "title": "Q1 2024 Supplier Performance Review",
        "content": (
            "Q1 2024 Performance Review Summary: Acme Foods maintained 98% on-time delivery rate, "
            "with quality score of 9.2/10. Customer satisfaction: 4.8/5 stars. "
            "Globex Dairy improved quality from 94% to 97%, on-time delivery 95%. "
            "Excellent communication and responsive to special requests. "
            "Sysco maintained competitive pricing with 96% delivery reliability. "
            "Quality score: 9.0/10. Large variety of product specifications available. "
            "US Foods expanded delivery zones to cover all district schools. "
            "On-time delivery: 92%, quality: 8.8/10. Good value for budget-conscious procurement. "
            "Overall district satisfaction: 8.5/10 across all categories. "
            "Recommendations: Continue contracts with Acme and Globex; negotiate volume discounts "
            "with Sysco; evaluate US Foods for secondary supplier relationship."
        ),
        "metadata": {
            "supplier_name": "Riverside USD", "category": "Performance Review",
            "bid_date": "2024-04-01", "document_type": "Internal Report",
            "status": "Active", "bid_id": "BID-2024-089",
        },
    },
}

upsert_documents(SOURCE_DOCS)

## Step 7b — Optional: pull documents from a table you already have

If bid/RFP text already lives elsewhere in this database, map it into `rag_documents` rather than
retyping it. This is the normal production path — one small adapter per source system.

In [ ]:
def import_from_existing_table(table: str, id_col: str, text_col: str,
                               title_col: str = None,
                               metadata_cols: list = None,
                               where: str = None, limit: int = None):
    """Copy rows from an existing table into rag_documents.

    metadata_cols become JSONB keys, so they are usable as retrieval filters *and*
    appear in the contextual header when they match HEADER_FIELDS.
    """
    metadata_cols = metadata_cols or []
    cols = [id_col, text_col] + ([title_col] if title_col else []) + metadata_cols
    sql = f"SELECT {', '.join(cols)} FROM {table}"
    if where:
        sql += f" WHERE {where}"
    if limit:
        sql += f" LIMIT {int(limit)}"

    with db(dict_rows=True) as cur:
        cur.execute(sql)
        rows = cur.fetchall()

    doc_map = {}
    for r in rows:
        if not r[text_col]:
            continue                       # nothing to chunk
        doc_id = str(r[id_col])
        doc_map[doc_id] = {
            "title": str(r[title_col]) if title_col and r[title_col] else doc_id,
            "content": r[text_col],
            "metadata": {c: (str(r[c]) if r[c] is not None else None) for c in metadata_cols},
        }
    if doc_map:
        upsert_documents(doc_map)
    return len(doc_map)

# Example - point it at your real table and uncomment:
# import_from_existing_table("bids", "bid_id", "description",
#                            title_col="bid_name",
#                            metadata_cols=["supplier_name", "category", "status"],
#                            where="status = 'Active'")
print("import_from_existing_table() ready.")

## Step 7c — Incremental ingest

v1.1 dropped and rebuilt every table on every run — 27 combinations, every chunk re-embedded, every
time. Fine for six demo documents; ruinous on a real corpus (cost, wall-clock, and rate limits).

Here a document is re-chunked and re-embedded **only when its `content_hash` differs from the
`indexed_hash` we last embedded**. Everything else is skipped. The chunk rewrite is one transaction per
document, so a crash mid-ingest can never leave a document half-indexed.

In [ ]:
def ensure_vector_index():
    """Build the HNSW index once, after the bulk load.

    m / ef_construction trade build time and memory against recall. m=16, ef_construction=64
    is the standard starting point; raise ef_construction toward 128 when recall matters more
    than build time. Requires EMBED_DIM <= 2000.
    """
    with db() as cur:
        cur.execute(f"""
            CREATE INDEX IF NOT EXISTS {CHUNK_TABLE}_embedding_hnsw
            ON {CHUNK_TABLE} USING hnsw (embedding vector_cosine_ops)
            WITH (m = {HNSW_M}, ef_construction = {HNSW_EF_CONSTRUCTION})
        """)
        cur.execute(f"ANALYZE {CHUNK_TABLE}")

def ingest(force: bool = False, only: list = None) -> dict:
    """Chunk + embed + store every document whose content changed since it was last indexed.

    force=True -> re-embed everything regardless. You MUST do this after changing
                  EMBED_MODEL, EMBED_DIM, or the chunking settings, because the vectors
                  already stored were produced by different rules and are now invalid.
    only=[ids] -> restrict to specific documents.
    """
    # Build the WHERE clause. "IS DISTINCT FROM" is like != but treats NULL sensibly:
    # a plain "indexed_hash != content_hash" would be NULL (not true!) for never-indexed
    # documents, silently skipping every new document you add. A classic SQL trap.
    clause = "TRUE" if force else "indexed_hash IS DISTINCT FROM content_hash"
    params = []
    if only:
        clause += " AND doc_id = ANY(%s)"    # ANY(array) is the Postgres way to say "IN (list)"
        params.append(only)

    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT doc_id, title, content, metadata, content_hash "
                    f"FROM {DOC_TABLE} WHERE {clause} ORDER BY doc_id", params)
        pending = cur.fetchall()             # only the documents that actually need work

    if not pending:
        print("Nothing to ingest - every document is already up to date.")
        ensure_vector_index()                # still make sure the index exists
        return {"documents": 0, "chunks": 0}

    total_chunks = 0
    for doc in pending:
        # 1. Cut the document into chunks, each carrying its contextual header (Step 6).
        chunks = build_chunks(doc["doc_id"], doc["title"], doc["content"], doc["metadata"])

        # 2. Embed the HEADER+TEXT version, not the clean text. This is the fix for your bug:
        #    the supplier/category/date become part of what the vector represents.
        vectors = embed_texts([c["embed_input"] for c in chunks], "RETRIEVAL_DOCUMENT")

        # 3. Pair each chunk with its vector. zip() walks both lists together.
        #    str(v) turns [0.1, 0.2, ...] into "[0.1, 0.2, ...]", which ::vector parses.
        rows = [
            (doc["doc_id"], c["chunk_index"], c["content"], c["embed_input"],
             c["token_count"], json.dumps(doc["metadata"]), str(v))
            for c, v in zip(chunks, vectors)
        ]

        # 4. Replace this document's chunks. All three statements are in ONE transaction
        #    (one `with db()` block), so either all of it lands or none of it does - a crash
        #    here can never leave a document with its old chunks deleted and no new ones.
        with db() as cur:
            # Delete-then-insert rather than update: the new chunk COUNT may differ from the
            # old one, so there is no row-by-row correspondence to update.
            cur.execute(f"DELETE FROM {CHUNK_TABLE} WHERE doc_id = %s", (doc["doc_id"],))
            # execute_values sends all rows in one round trip instead of one INSERT per chunk.
            # `template` tells it how to render each row, including the ::jsonb / ::vector casts.
            execute_values(cur, f"""
                INSERT INTO {CHUNK_TABLE}
                    (doc_id, chunk_index, content, embed_input, token_count, metadata, embedding)
                VALUES %s
            """, rows, template="(%s, %s, %s, %s, %s, %s::jsonb, %s::vector)", page_size=200)
            # Advance the watermark: "the vectors on disk now reflect this version of the text".
            # Because this is inside the same transaction, it can only be recorded if the
            # chunks actually landed - so the two can never disagree.
            cur.execute(f"UPDATE {DOC_TABLE} SET indexed_hash = %s WHERE doc_id = %s",
                        (doc["content_hash"], doc["doc_id"]))

        total_chunks += len(rows)
        avg_tokens = sum(c["token_count"] for c in chunks) // max(1, len(chunks))
        print(f"  {doc['doc_id']:<28} {len(rows):>3} chunks (avg {avg_tokens} tokens)")

    ensure_vector_index()
    print(f"\nIngested {len(pending)} document(s), {total_chunks} chunks. HNSW index ready.")
    return {"documents": len(pending), "chunks": total_chunks}

stats = ingest()

with db(dict_rows=True) as cur:
    cur.execute(f"""
        SELECT count(*) AS chunks, count(DISTINCT doc_id) AS docs,
               round(avg(token_count)) AS avg_tokens, max(token_count) AS max_tokens,
               pg_size_pretty(pg_total_relation_size('{CHUNK_TABLE}')) AS on_disk
        FROM {CHUNK_TABLE}
    """)
    print("\nIndex state:", dict(cur.fetchone()))

## Step 8 — Hybrid retrieval (vector + full-text) fused with RRF

The core upgrade. Two independent retrievers run in **one** SQL statement and their ranked lists are
merged with **Reciprocal Rank Fusion**:

```
score(chunk) = Σ_retrievers  weight_r / (k + rank_r(chunk)),   k = 60
```

RRF fuses on *rank*, never on raw score — so a cosine distance and a `ts_rank_cd` score never have to be
put on a common scale. That scale mismatch is what makes naive score-blended hybrid search fragile.

Why both legs earn their keep:

| Query | Dense alone | Lexical alone | Hybrid |
|---|---|---|---|
| *"who delivers fastest?"* | ✅ semantic paraphrase | ❌ no term overlap | ✅ |
| *"BID-2024-089"* | ⚠️ IDs embed poorly | ✅ exact token | ✅ |
| *"Acme customer satisfaction"* | ⚠️ | ✅ header carries "Acme" | ✅ |

Three details that matter and are easy to get wrong:

- **The KNN sits in an inner subquery.** A window function over `ORDER BY embedding <=> ...` at the same
  query level forces Postgres to rank *every* row and throws away the index. Ranking the already-limited
  40 rows in an outer query keeps the HNSW scan.
- **`hnsw.ef_search`** is set per transaction — the recall/latency dial you actually tune in production.
- **JSONB metadata pre-filtering** (`metadata @> filter`) is evaluated inside the ANN scan.

Both `vec_rank` and `txt_rank` come back, so when debugging you can see *which* leg found each chunk.

In [ ]:
# =====================================================================================
# One SQL statement runs BOTH searches and merges them. Reading guide:
#   WITH name AS (...)  = a "CTE", a named temporary result you can refer to below.
#                         Think of it as a variable holding a table.
#   %(name)s            = a placeholder. psycopg2 substitutes the value safely -
#                         this is what prevents SQL injection. NEVER build SQL with
#                         f-strings around user input; table names only, as here.
# The three CTEs are: vec (meaning search) -> txt (keyword search) -> fused (merge).
# =====================================================================================
HYBRID_SQL = f"""
-- ============ CTE 1: the MEANING search (dense / vector / ANN) ============
WITH vec AS (
    -- Why the nested SELECT: a window function like ROW_NUMBER() at the SAME level as
    -- "ORDER BY embedding <=> ..." forces Postgres to rank EVERY row in the table,
    -- which throws away the HNSW index and turns this into a full scan. So the inner
    -- query does the indexed nearest-neighbour lookup and takes the top N; the outer
    -- query then numbers only those N surviving rows. This one detail is the
    -- difference between a millisecond and a full table scan.
    SELECT chunk_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
    FROM (
        SELECT chunk_id, embedding <=> %(qv)s::vector AS distance
        FROM {CHUNK_TABLE}
        -- The metadata filter. If no filter was passed we send NULL, and
        -- "NULL IS NULL" is true, so the whole condition passes and nothing is excluded.
        -- @> means "contains": metadata @> '{{"category":"Dairy"}}' matches rows whose
        -- metadata includes that key/value. Backed by the GIN index from Step 4.
        WHERE (%(filter)s::jsonb IS NULL OR metadata @> %(filter)s::jsonb)
        ORDER BY embedding <=> %(qv)s::vector   -- <=> is cosine distance; ASC = nearest first
        LIMIT %(vec_limit)s
    ) v
),
-- ============ CTE 2: the KEYWORD search (lexical / full-text) ============
txt AS (
    SELECT chunk_id, ROW_NUMBER() OVER (ORDER BY score DESC) AS rank
    FROM (
        -- websearch_to_tsquery parses a normal user query the way a search engine would
        -- (it understands quotes and OR), so you can pass raw user input straight in.
        -- The comma is an implicit CROSS JOIN: it just names that parsed query as `q`.
        -- ts_rank_cd scores how well a row matches, factoring in how CLOSE the matched
        -- words are to each other ("cover density") - not merely how often they appear.
        SELECT c.chunk_id, ts_rank_cd(c.tsv, q) AS score
        FROM {CHUNK_TABLE} c, websearch_to_tsquery('english', %(question)s) q
        WHERE c.tsv @@ q        -- @@ means "this text matches this query"
          AND (%(filter)s::jsonb IS NULL OR c.metadata @> %(filter)s::jsonb)
        ORDER BY score DESC     -- keyword scores are "higher is better", unlike distance
        LIMIT %(txt_limit)s
    ) t
),
-- ============ CTE 3: merge the two ranked lists (Reciprocal Rank Fusion) ============
fused AS (
    SELECT
        -- FULL OUTER JOIN keeps chunks found by EITHER search, so a chunk that only the
        -- keyword search found is not discarded. But that means v.chunk_id is NULL for
        -- those rows, hence COALESCE (= "first non-NULL value") to get a usable id.
        COALESCE(v.chunk_id, t.chunk_id) AS chunk_id,

        -- The RRF formula: each search contributes weight / (k + its_rank).
        -- rank 1 -> 1/61 = 0.0164,  rank 2 -> 1/62 = 0.0161,  rank 40 -> 1/100 = 0.0100.
        -- A chunk both searches ranked highly beats a chunk only one of them loved.
        -- COALESCE(..., 0) means "found by only one search" scores 0 from the other,
        -- rather than making the whole sum NULL.
        COALESCE(%(w_vec)s / (%(rrf_k)s + v.rank), 0)
      + COALESCE(%(w_txt)s / (%(rrf_k)s + t.rank), 0) AS rrf_score,

        v.rank AS vec_rank,   -- kept for debugging: WHICH search found this chunk, and where?
        t.rank AS txt_rank    -- NULL here = the keyword search did not find it at all
    FROM vec v FULL OUTER JOIN txt t ON v.chunk_id = t.chunk_id
)
-- ============ Final: attach the actual text and document title ============
-- The CTEs only carried ids and scores around (cheap). Now we join back to fetch the
-- content for the handful of rows that actually won.
SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, c.metadata,
       d.title, f.rrf_score, f.vec_rank, f.txt_rank,
       (c.embedding <=> %(qv)s::vector) AS cosine_distance   -- reported so we can apply a relevance floor
FROM fused f
JOIN {CHUNK_TABLE} c ON c.chunk_id = f.chunk_id
JOIN {DOC_TABLE}   d ON d.doc_id   = c.doc_id
ORDER BY f.rrf_score DESC, cosine_distance ASC   -- best fused score first; distance breaks ties
LIMIT %(limit)s
"""

def retrieve(question: str, qv=None, mode: str = "hybrid", top_k: int = TOP_K,
             filters: dict = None):
    """First-stage retrieval: find candidate chunks.

    question - the user's question, used by the keyword leg verbatim
    qv       - a pre-computed query embedding. Passing it in lets Step 11 time the
               embedding and the search separately instead of lumping them together
               (the mistake that made v1.1's benchmark meaningless).
    mode     - "hybrid" (both searches) | "vector" (meaning only) | "text" (keywords only).
               The single-leg modes exist so Step 11 can PROVE hybrid earns its complexity
               rather than asserting it.
    filters  - metadata restriction, e.g. {"category": "Dairy"}.
    """
    if qv is None:
        qv = list(embed_query_cached(question))   # cached tuple -> list

    # One switchboard drives all three modes: disabling a search leg is just setting its
    # LIMIT to 0 and its weight to 0. Fewer code paths = fewer places for a bug to hide.
    params = {
        "qv": str(list(qv)),   # pgvector accepts the Python list's string form: "[0.1, 0.2, ...]"
        "question": question,
        # json.dumps turns {"category":"Dairy"} into the string '{"category":"Dairy"}';
        # None becomes SQL NULL, which the "IS NULL OR ..." check treats as "no filter".
        "filter": json.dumps(filters) if filters else None,
        "vec_limit": VECTOR_CANDIDATES if mode in ("hybrid", "vector") else 0,
        "txt_limit": TEXT_CANDIDATES   if mode in ("hybrid", "text")   else 0,
        "w_vec":     W_VECTOR if mode in ("hybrid", "vector") else 0.0,
        "w_txt":     W_TEXT   if mode in ("hybrid", "text")   else 0.0,
        "rrf_k": RRF_K,
        "limit": top_k,
    }
    with db(dict_rows=True) as cur:
        # ef_search = how hard HNSW looks before settling. Higher = better recall, slower.
        # SET LOCAL confines it to THIS transaction, so one expensive query cannot slow
        # everyone else down. Postgres does not allow placeholders in SET, so the value is
        # interpolated - int() makes that safe by guaranteeing it is a number, not text.
        cur.execute(f"SET LOCAL hnsw.ef_search = {int(HNSW_EF_SEARCH)}")
        cur.execute(HYBRID_SQL, params)
        return [dict(r) for r in cur.fetchall()]   # RealDictRow -> plain dict

# Side by side on the exact query v1.1 got wrong.
q = "What was the Customer Satisfaction level for supplier performance review?"
for mode in ("vector", "text", "hybrid"):
    print(f"\n--- {mode} ---")
    for i, h in enumerate(retrieve(q, mode=mode, top_k=3), 1):
        print(f"{i}. {h['doc_id']:<26} vec={h['vec_rank']} txt={h['txt_rank']} "
              f"dist={h['cosine_distance']:.3f}\n   {h['content'][:110]}...")

## Step 9 — Reranking

First-stage retrieval optimises for *recall* — cast a wide net (40 candidates, 20 kept). Reranking
optimises for *precision* — a slower, smarter model reads the query against each candidate and
reorders. This is the highest-leverage single component in most production RAG systems, and v1.1 had
none: its 3 vector hits went straight to the LLM.

**In production, prefer a dedicated cross-encoder.** On Google Cloud that is the Vertex AI Ranking API
(`semantic-ranker-default@latest`) — purpose-built, and far cheaper and lower-latency than an LLM call.
The LLM reranker below is the portable equivalent so this notebook runs with no extra service enabled;
`rerank()` is a drop-in seam — replace the body, keep the signature.

Note the fallback: if the reranker errors or returns malformed JSON we keep the RRF order rather than
failing the request. A reranker outage should degrade quality, not take retrieval down.

In [ ]:
from google.genai.types import GenerateContentConfig

RERANK_PROMPT = """You are a search relevance rater for a procurement document system.

Rate how well each passage answers the QUESTION, on a 0-10 scale:
  10 = contains the exact answer
   7 = directly about the question's subject, partial answer
   4 = same topic, does not answer the question
   0 = irrelevant

Judge each passage independently. Return ONLY a JSON array of objects with keys
"id" (the passage number) and "score" (integer 0-10). No prose.

QUESTION: {question}

PASSAGES:
{passages}"""

def rerank(question: str, hits: list, top_n: int = RERANK_TOP_N):
    """Second-stage precision pass. Returns at most top_n hits, each with 'rerank_score'.

    Production swap - Vertex AI Ranking API:
        from google.cloud import discoveryengine_v1 as de
        client.rank(ranking_config=..., model="semantic-ranker-default@latest", records=[...])
    """
    if not hits:
        return hits
    passages = "\n\n".join(
        f"[{i}] ({h['title']}) {h['content'][:1200]}" for i, h in enumerate(hits)
    )
    try:
        resp = _with_retry(lambda: genai_client.models.generate_content(
            model=RERANK_MODEL,
            contents=RERANK_PROMPT.format(question=question, passages=passages),
            config=GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        ))
        scores = {int(r["id"]): float(r["score"]) for r in json.loads(resp.text)}
    except Exception as e:
        # Degrade to first-stage order rather than failing the whole query.
        print(f"  [rerank unavailable, keeping RRF order: {type(e).__name__}]")
        return hits[:top_n]

    for i, h in enumerate(hits):
        h["rerank_score"] = scores.get(i, 0.0)
    ranked = sorted(hits, key=lambda h: h["rerank_score"], reverse=True)
    # Drop candidates the reranker judged off-topic even when there is room for them:
    # padding the context with irrelevant passages measurably degrades the answer.
    return [h for h in ranked if h["rerank_score"] >= 4][:top_n] or ranked[:1]

_hits = retrieve(q, mode="hybrid", top_k=RERANK_CANDIDATES)
print(f"{len(_hits)} candidates -> reranked:\n")
for h in rerank(q, _hits):
    print(f"{h.get('rerank_score', '-'):>5}  {h['doc_id']:<26} {h['content'][:90]}...")

## Step 10 — Grounded generation with citations

The prompt is where your wrong answer was actually produced, so it carries explicit rules:

1. **Cite every fact** with a source number — an unverifiable answer becomes impossible.
2. **List *all* matching entities** — the direct fix for `4.8/5` vs `8.5/10`. The review holds one
   per-supplier satisfaction figure and one district-wide figure; a good answer surfaces both, labelled,
   instead of silently choosing.
3. **Answer for the entity that was asked about** — if you ask about Acme and the context only covers the
   district, the model must say so rather than substituting a different number.
4. **Quote figures exactly** — no rounding, no unit conversion, no "approximately".
5. **A fixed refusal string** — machine-detectable, so you can alarm on refusal rate in production.

Plus a **relevance floor**: if nothing clears `MAX_COSINE_DISTANCE` and no lexical leg fired, we refuse
before spending a generation call. v1.1 always returned its top 3 no matter how irrelevant — which is
exactly how RAG systems end up confidently answering out of unrelated documents.

Each source block also carries its metadata, so the model can attribute a bare number to a supplier.

In [ ]:
REFUSAL = "I don't have that in the indexed documents."

SYSTEM_RULES = f"""You are a procurement analyst assistant. Answer using ONLY the numbered SOURCES below.

Rules:
1. Cite every fact with its source number in square brackets, e.g. [2]. No citation, no claim.
2. If several entities or several values legitimately match the question, list ALL of them with
   attribution. Never collapse distinct figures into one number, and never present an aggregate as
   though it were an individual entity's value (or the reverse).
3. If the question names a specific entity (supplier, bid, category), answer for THAT entity. If the
   sources only cover a different entity, say so explicitly instead of substituting it.
4. Quote figures exactly as written in the sources - same units, scale, currency and precision.
5. If the sources do not contain the answer, reply with exactly: "{REFUSAL}"
   Do not guess, and do not use knowledge from outside the sources.
6. Be concise. Lead with the answer, then the supporting detail."""

def format_sources(hits) -> str:
    """Turn retrieved chunks into the numbered [1] [2] [3] block the prompt refers to.

    Each block carries the document title AND its metadata, so the model can attribute a
    bare number like "4.8/5" to the right supplier. Without this the model sees floating
    facts with no owner - which is precisely how v1.1 produced the wrong answer.
    """
    blocks = []
    for i, h in enumerate(hits, 1):        # enumerate(..., 1) numbers from 1, not 0
        m = h.get("metadata") or {}        # `or {}` guards against metadata being None
        facets = " | ".join(f"{k}: {m[k]}" for k in HEADER_FIELDS if m.get(k))
        head = f"[{i}] {h['title']} (doc_id: {h['doc_id']}" + (f" | {facets}" if facets else "") + ")"
        blocks.append(f"{head}\n{h['content']}")
    return "\n\n".join(blocks)

def ask(question: str, filters: dict = None, mode: str = "hybrid",
        use_reranker: bool = USE_RERANKER, verbose: bool = False) -> dict:
    """The full RAG request. Four stages, each timed separately:

        1. embed    - turn the question into a vector
        2. search   - find candidate chunks (Step 8)
        3. rerank   - keep only the best few (Step 9)
        4. generate - Gemini writes the answer from those chunks only

    Returns a dict with the answer, the sources behind it, and per-stage timings - so
    every answer is auditable and every slow request is diagnosable.

    verbose=True prints the assembled prompt. Do this once: seeing exactly what the model
    was given is the fastest way to understand any wrong answer.
    """
    timings = {}

    # ---- Stage 1: embed the question ----
    t0 = time.perf_counter()               # perf_counter is a high-resolution stopwatch
    qv = list(embed_query_cached(question))
    timings["embed_ms"] = (time.perf_counter() - t0) * 1000   # seconds -> milliseconds

    # ---- Stage 2: search ----
    # Retrieve MORE candidates when reranking, because the reranker's job is to sift a wide
    # net. Without one, take only what will fit in the prompt.
    first_stage_k = RERANK_CANDIDATES if use_reranker else TOP_K
    t0 = time.perf_counter()
    hits = retrieve(question, qv=qv, mode=mode, top_k=first_stage_k, filters=filters)
    timings["search_ms"] = (time.perf_counter() - t0) * 1000

    # ---- Guardrail: refuse BEFORE paying for a generation call ----
    # Two conditions must BOTH hold to refuse: nothing was semantically close, AND no
    # keyword matched either (txt_rank is None means the keyword search never found it).
    # Requiring both avoids refusing on a valid exact-ID lookup, where cosine distance can
    # look poor but the keyword hit is decisive.
    # v1.1 had no such check: it always returned its top 3, however irrelevant, which is
    # how RAG systems end up confidently answering from unrelated documents.
    if not hits or (min(h["cosine_distance"] for h in hits) > MAX_COSINE_DISTANCE
                    and all(h["txt_rank"] is None for h in hits)):
        return {"answer": REFUSAL, "sources": [], "timings": timings, "refused": True}

    # ---- Stage 3: rerank ----
    t0 = time.perf_counter()
    hits = rerank(question, hits) if use_reranker else hits[:RERANK_TOP_N]
    timings["rerank_ms"] = (time.perf_counter() - t0) * 1000

    # ---- Stage 4: build the prompt and generate ----
    # Structure: rules first, then the evidence, then the question. The model is instructed
    # to use ONLY what appears between them.
    prompt = f"{SYSTEM_RULES}\n\nSOURCES:\n{format_sources(hits)}\n\nQUESTION: {question}"
    if verbose:
        print(prompt[:2000], "\n...\n")

    t0 = time.perf_counter()
    resp = _with_retry(lambda: genai_client.models.generate_content(
        model=CHAT_MODEL, contents=prompt,
        # temperature=0 = be as deterministic as possible. Creativity is exactly what you
        # do NOT want here; you want faithful reporting of the sources.
        config=GenerateContentConfig(temperature=0, max_output_tokens=1024),
    ))
    timings["generate_ms"] = (time.perf_counter() - t0) * 1000

    answer = (resp.text or "").strip()
    return {
        "answer": answer,
        "sources": [{"n": i, "doc_id": h["doc_id"], "title": h["title"],
                     "content": h["content"], "metadata": h.get("metadata") or {},
                     "cosine_distance": round(h["cosine_distance"], 4),
                     "rerank_score": h.get("rerank_score")}
                    for i, h in enumerate(hits, 1)],
        "timings": timings,
        "refused": answer.startswith(REFUSAL),
    }

def show(result: dict):
    print(result["answer"])
    print("\nSources:")
    for s in result["sources"]:
        print(f"  [{s['n']}] {s['doc_id']} (dist={s['cosine_distance']}, "
              f"rerank={s['rerank_score']})")
    print("  timings:", {k: round(v) for k, v in result["timings"].items()}, "ms")

### The regression test: your original question

Three runs. The first is verbatim the question v1.1 answered wrongly — it should now surface **both**
satisfaction figures with attribution and citations. The third is not in the corpus at all, so it must
refuse rather than invent.

In [ ]:
show(ask("What was the Customer Satisfaction level for supplier performance review?"))
print("\n" + "=" * 90 + "\n")
show(ask("What was Acme Foods' customer satisfaction rating?"))
print("\n" + "=" * 90 + "\n")
show(ask("What is Acme's organic produce certification policy?"))   # not in the corpus -> refuse

In [ ]:
# Exact-identifier lookup - the case pure dense retrieval loses and hybrid wins.
show(ask("What are the evaluation criteria weightings for BID-2024-089?"))
print("\n" + "=" * 90 + "\n")
# Metadata pre-filtering: constrain the ANN scan before it runs.
show(ask("What are the payment terms?", filters={"category": "Dairy"}))

## Step 11 — Evaluation that means something

Three flaws made v1.1's leaderboard unusable:

1. **Latency included the ~3.2 s embedding call**, drowning the sub-millisecond Postgres search it was
   meant to measure. Every index type scored ~3.2 s → no signal → it crowned a brute-force scan.
2. **Doc-level gold only.** "The right document was in the top 3" does not tell you whether the chunk
   holding the *answer* was retrieved. Your satisfaction question retrieved the right document and still
   produced a wrong answer — v1.1's metrics would have scored that a perfect 1.0.
3. **No unanswerable questions**, so hallucination was never measured at all.

v2 addresses all three:

- **`embed_ms` / `search_ms` / `rerank_ms` reported separately** — `search_ms` is now the number you tune.
- **`answer_support@k`** — did a retrieved chunk actually contain the answer string? This is the metric
  that would have caught your bug.
- **nDCG@k** alongside recall/MRR, and a **refusal-correctness** check in Step 12.
- **Corrected gold labels** — `"Which suppliers offered an early payment discount?"` now lists all three
  that do. v1.1 was marking two correct retrievals as failures.

All configurations are scored on the same final context size (`RERANK_TOP_N`), so reranked and
non-reranked runs are actually comparable.

13 questions is still a demo. **Before you deploy, write 50–100 questions from real user logs.** That is
the highest-value hour you can spend on a RAG system — nothing else tells you whether a change helped.

In [ ]:
# doc_ids: every document that legitimately answers the question.
# must_contain: the literal string that must appear in a retrieved chunk for the answer to be
#               derivable - this is what catches "right document, wrong chunk".
GOLD = [
    {"q": "What did Acme quote for romaine lettuce?",
     "doc_ids": ["acme-produce-2024"], "must_contain": ["3.20"]},
    {"q": "What was Acme Foods' customer satisfaction rating in the Q1 performance review?",
     "doc_ids": ["performance-review-2024"], "must_contain": ["4.8/5"]},
    {"q": "What is the overall district satisfaction score across all categories?",
     "doc_ids": ["performance-review-2024"], "must_contain": ["8.5/10"]},
    {"q": "Which suppliers offered an early payment discount?",
     "doc_ids": ["globex-dairy-2024", "sysco-proteins-2024", "us-foods-beverages-2024"],
     "must_contain": ["early payment discount"]},
    {"q": "What were Globex's payment terms?",
     "doc_ids": ["globex-dairy-2024"], "must_contain": ["Net-45"]},
    {"q": "What was the supplier response rate?",
     "doc_ids": ["bid-summary-2024"], "must_contain": ["62.5%"]},
    {"q": "When is the customer due date?",
     "doc_ids": ["bid-summary-2024"], "must_contain": ["2024-05-25"]},
    {"q": "How much does Sysco charge for wild-caught salmon fillets?",
     "doc_ids": ["sysco-proteins-2024"], "must_contain": ["12.80"]},
    {"q": "BID-2024-089 evaluation criteria",
     "doc_ids": ["bid-summary-2024"], "must_contain": ["40%"]},
    {"q": "Which supplier had the lowest on-time delivery rate?",
     "doc_ids": ["performance-review-2024"], "must_contain": ["92%"]},
    {"q": "What is the minimum order value for beverages?",
     "doc_ids": ["us-foods-beverages-2024"], "must_contain": ["$250"]},
    {"q": "Who guarantees cold chain on dairy deliveries?",
     "doc_ids": ["globex-dairy-2024"], "must_contain": ["Cold chain"]},
    # Unanswerable - the system must refuse, not invent.
    {"q": "What is Acme's organic produce certification policy?",
     "doc_ids": [], "must_contain": [], "unanswerable": True},
]

print(f"{len(GOLD)} evaluation questions "
      f"({sum(1 for g in GOLD if g.get('unanswerable'))} unanswerable).")

In [ ]:
def _ndcg(ranked_docs, gold_docs, k):
    """Binary-relevance nDCG@k - rewards ranking a correct doc first, not merely including it."""
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(ranked_docs[:k]) if d in gold_docs)
    ideal = sum(1.0 / math.log2(i + 2) for i in range(min(len(gold_docs), k)))
    return dcg / ideal if ideal else 0.0

def evaluate(mode: str = "hybrid", use_reranker: bool = False, k: int = RERANK_TOP_N):
    """Retrieval-quality metrics with per-stage latency.

    Uses the UNCACHED embedder so embed_ms reflects a real cold request. Every config is
    scored on the same final list size k, so reranked and non-reranked runs compare fairly.
    """
    answerable = [g for g in GOLD if not g.get("unanswerable")]
    recall = mrr = ndcg = support = 0.0
    t_embed = t_search = t_rerank = 0.0

    for g in answerable:
        t0 = time.perf_counter()
        qv = embed_query(g["q"])
        t_embed += (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        hits = retrieve(g["q"], qv=qv, mode=mode,
                        top_k=RERANK_CANDIDATES if use_reranker else k)
        t_search += (time.perf_counter() - t0) * 1000

        if use_reranker:
            t0 = time.perf_counter()
            hits = rerank(g["q"], hits, top_n=k)
            t_rerank += (time.perf_counter() - t0) * 1000
        hits = hits[:k]

        ranked = [h["doc_id"] for h in hits]
        gold_docs = set(g["doc_ids"])
        if gold_docs & set(ranked):
            recall += 1
            first = next(i for i, d in enumerate(ranked) if d in gold_docs)
            mrr += 1.0 / (first + 1)
        ndcg += _ndcg(ranked, gold_docs, k)
        # answer_support: is the answer string actually present in what we retrieved?
        blob = " ".join(h["content"] for h in hits).lower()
        if all(s.lower() in blob for s in g["must_contain"]):
            support += 1

    n = len(answerable)
    return {
        "mode": mode + (" + rerank" if use_reranker else ""),
        "recall": recall / n, "mrr": mrr / n, "ndcg": ndcg / n,
        "support": support / n,
        "embed_ms": t_embed / n, "search_ms": t_search / n,
        "rerank_ms": (t_rerank / n) if use_reranker else 0.0,
    }

rows = []
for mode, rr in [("text", False), ("vector", False), ("hybrid", False), ("hybrid", True)]:
    r = evaluate(mode, rr)
    rows.append(r)
    print(f"  done: {r['mode']}")

k = RERANK_TOP_N
hdr = (f"{'configuration':<20}{'recall@' + str(k):>10}{'mrr':>8}{'ndcg@' + str(k):>9}"
       f"{'support':>9}{'embed ms':>10}{'search ms':>11}{'rerank ms':>11}")
print("\n" + hdr)
print("-" * len(hdr))
for r in rows:
    print(f"{r['mode']:<20}{r['recall']:>10.2f}{r['mrr']:>8.2f}{r['ndcg']:>9.2f}"
          f"{r['support']:>9.2f}{r['embed_ms']:>10.0f}{r['search_ms']:>11.1f}"
          f"{r['rerank_ms']:>11.0f}")

print("\n`search ms` is now its own column - a few milliseconds of Postgres work that v1.1 was")
print("reporting as ~3200 ms because it timed the embedding API call alongside it.")
print("On six documents most configs will tie; the spread appears at real corpus size, and the")
print("exact-ID and multi-entity questions are where hybrid and reranking already separate.")

## Step 12 — Answer-quality evaluation (LLM-as-judge)

Retrieval metrics tell you whether the right text was *found*. They cannot tell you whether the answer
was *right* — your satisfaction question is the proof: retrieval was fine, the answer was not.

So the second harness scores the generated answers on three axes an independent model can judge:

- **groundedness** — is every claim supported by the retrieved sources? (hallucination detector)
- **correctness** — does it contain the expected fact?
- **completeness** — when several entities match, are they all covered? (the exact v1.1 failure)

Plus a deterministic `exact_fact_present` backstop, and a hard **refusal check** on the unanswerable
question. An LLM judge is noisy on any single item — use it to compare *configurations across a whole
set*, and keep a small human-labelled slice as ground truth.

In [ ]:
JUDGE_PROMPT = """You are evaluating a retrieval-augmented answer. Be strict.

QUESTION: {question}
EXPECTED FACT(S) a correct answer must contain: {expected}

SOURCES THAT WERE RETRIEVED:
{sources}

ANSWER GIVEN:
{answer}

Score 0-5 on each axis:
- groundedness: every claim is supported by the sources (5 = fully grounded, 0 = fabricated)
- correctness: the answer contains the expected fact(s), stated accurately
- completeness: if several entities or values legitimately match, all are covered and attributed

Return ONLY JSON: {{"groundedness": int, "correctness": int, "completeness": int, "why": "one sentence"}}"""

def judge_answers(mode: str = "hybrid", use_reranker: bool = True):
    scored, refusals_ok, refusals_total = [], 0, 0

    for g in GOLD:
        res = ask(g["q"], mode=mode, use_reranker=use_reranker)

        if g.get("unanswerable"):
            refusals_total += 1
            ok = res["refused"]
            refusals_ok += ok
            print(f"{'PASS' if ok else 'FAIL'}  (refusal) {g['q'][:60]}")
            continue

        src = format_sources(res["sources"]) or "(none)"
        try:
            jr = _with_retry(lambda: genai_client.models.generate_content(
                model=CHAT_MODEL,
                contents=JUDGE_PROMPT.format(question=g["q"],
                                             expected=", ".join(g["must_contain"]),
                                             sources=src, answer=res["answer"]),
                config=GenerateContentConfig(temperature=0,
                                             response_mime_type="application/json")))
            verdict = json.loads(jr.text)
        except Exception as e:
            print(f"  judge failed on {g['q'][:40]}: {type(e).__name__}: {e}")
            continue

        # Deterministic backstop: does the expected literal actually appear in the answer?
        verdict["exact"] = all(s.lower() in res["answer"].lower() for s in g["must_contain"])
        scored.append(verdict)
        print(f"g={verdict['groundedness']} c={verdict['correctness']} "
              f"comp={verdict['completeness']} exact={verdict['exact']}  {g['q'][:55]}")

    if scored:
        avg = lambda key: sum(s[key] for s in scored) / len(scored)
        print(f"\nGroundedness {avg('groundedness'):.2f}/5 | "
              f"Correctness {avg('correctness'):.2f}/5 | "
              f"Completeness {avg('completeness'):.2f}/5")
        print(f"Expected fact literally present: "
              f"{sum(s['exact'] for s in scored)}/{len(scored)}")
    if refusals_total:
        print(f"Correct refusals: {refusals_ok}/{refusals_total}")

judge_answers()

## Step 13 — Configuration sweep: *how you justify the choice*

**This is the cell you run before a client meeting.** It answers "why this chunker?" and "why this
embedding model?" with a table instead of an opinion.

It compares, on **your** documents and **your** questions:

- **5 chunking strategies** — all three of v1.1's (`fixed_400`, `sentence_500`, `token_256`) plus the
  recursive splitter at two sizes, so the comparison is honest rather than rigged toward the new one.
- **Several embedding models** — including the older generation, so "why not 004?" has a measured answer.
- **Several dimensions** — 3072 vs 1536 vs 768 via Matryoshka truncation. Directly answers "can we halve
  storage without hurting accuracy?"
- **Header on vs off** — an *ablation*: same everything, contextual header removed. This isolates how
  much the header actually contributes, which is how you defend it as a design decision rather than a
  hunch.

Each configuration gets its **own throwaway table** (`rag_sweep_*`), because vectors from different
models and dimensions cannot share one. They are dropped at the end — unlike v1.1, which dropped the
table it had just crowned the winner.

### Read the output honestly

The report prints a **95% confidence interval** next to each score. With 12 questions that interval is
roughly **±25 points** — so most configurations will be *statistically indistinguishable*, and the table
will say so. **That is the correct result, not a broken one.** It is the same reason v1.1's leaderboard
was full of `0.80, 0.80, 0.80` ties.

> **What to do with that:** treat the sweep as a screening tool. It reliably catches configurations that
> are *badly* wrong (a chunker that shreds your tables, a dimension too small for your domain). It
> cannot resolve a 3-point gap until you have ~50–100 questions. Report it that way and you will be
> credible; report a 3-point win from 12 questions as decisive and a sharp reviewer will take the whole
> analysis apart.

**Cost warning.** This re-embeds your whole corpus once per configuration. Six demo documents is
pennies; 10,000 real documents × 12 configs is a real bill. Start with `SWEEP_CONFIGS[:4]`, and sweep on
a representative *sample* of a large corpus rather than all of it.

In [ ]:
# =====================================================================================
# v1.1's three chunkers, reproduced verbatim so the comparison is fair rather than rigged.
# Keeping them here means you can show a client the exact methods that were considered.
# =====================================================================================

def chunk_fixed(text, size=400, overlap=60):
    """v1.1's 'fixed_400': slide a fixed-width window over raw CHARACTERS.
    Simple and predictable, but happily cuts mid-word and mid-number."""
    text = text.strip()
    step = max(1, size - overlap)          # how far the window advances each time
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

def chunk_sentence(text, size=500):
    """v1.1's 'sentence': pack whole sentences up to a CHARACTER budget.
    Respects sentence boundaries, but a character budget does not map cleanly to the
    model's token limit, so chunk sizes vary unpredictably in token terms."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text.strip()) if s.strip()]
    chunks, buf = [], ""
    for s in sents:
        if buf and len(buf) + len(s) > size:
            chunks.append(buf); buf = s    # flush the full chunk, start a new one
        else:
            buf = f"{buf} {s}".strip()     # room left - keep appending
    if buf:
        chunks.append(buf)
    return chunks

def chunk_tokens(text, size=256, overlap=40):
    """v1.1's 'token_256': slide a window over TOKENS.
    Guarantees the model's token limit is respected, but cuts mid-sentence."""
    toks = _enc.encode(text)
    step = max(1, size - overlap)
    return [_enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]

# The candidate chunkers. Each is a function: text -> list of text pieces.
CHUNKERS = {
    "fixed_400":     lambda t: chunk_fixed(t, 400, 60),      # v1.1
    "sentence_500":  lambda t: chunk_sentence(t, 500),       # v1.1
    "token_256":     lambda t: chunk_tokens(t, 256, 40),     # v1.1
    "recursive_350": lambda t: split_text(t, 350, 60),       # v2 default (Step 6)
    "recursive_600": lambda t: split_text(t, 600, 100),      # v2, larger chunks
}

# The configurations to compare. Trim this list to control cost - every entry re-embeds
# the whole corpus once.
#   model / dim -> which embedder, at what vector size
#   chunker     -> a key from CHUNKERS above
#   header      -> True = embed the contextual header (Step 6); False = the ablation
SWEEP_CONFIGS = [
    # --- vary the CHUNKER, holding the model fixed ("why this chunking method?") ---
    {"name": "gem001-1536 / fixed_400",     "model": "gemini-embedding-001", "dim": 1536, "chunker": "fixed_400",     "header": True},
    {"name": "gem001-1536 / sentence_500",  "model": "gemini-embedding-001", "dim": 1536, "chunker": "sentence_500",  "header": True},
    {"name": "gem001-1536 / token_256",     "model": "gemini-embedding-001", "dim": 1536, "chunker": "token_256",     "header": True},
    {"name": "gem001-1536 / recursive_350", "model": "gemini-embedding-001", "dim": 1536, "chunker": "recursive_350", "header": True},
    {"name": "gem001-1536 / recursive_600", "model": "gemini-embedding-001", "dim": 1536, "chunker": "recursive_600", "header": True},

    # --- vary the MODEL, holding the chunker fixed ("why not 004 / 005?") ---
    {"name": "text-emb-004 / recursive_350", "model": "text-embedding-004",  "dim": 768,  "chunker": "recursive_350", "header": True},
    {"name": "text-emb-005 / recursive_350", "model": "text-embedding-005",  "dim": 768,  "chunker": "recursive_350", "header": True},

    # --- vary the DIMENSION ("can we halve storage?") ---
    {"name": "gem001-3072 / recursive_350", "model": "gemini-embedding-001", "dim": 3072, "chunker": "recursive_350", "header": True},
    {"name": "gem001-768  / recursive_350", "model": "gemini-embedding-001", "dim": 768,  "chunker": "recursive_350", "header": True},

    # --- ABLATION: identical to the v2 default, header removed ("does the header help?") ---
    {"name": "gem001-1536 / recursive_350 / NO HEADER", "model": "gemini-embedding-001", "dim": 1536,
     "chunker": "recursive_350", "header": False},
]

print(f"{len(SWEEP_CONFIGS)} configurations defined, {len(CHUNKERS)} chunkers available.")
print("Each config re-embeds the whole corpus once - trim SWEEP_CONFIGS to control cost.")

In [ ]:
# =====================================================================================
# The sweep machinery. Each config gets its own table because vectors from different
# models/dimensions are not comparable and cannot share a column.
# =====================================================================================

_sweep_qcache = {}   # (model, dim, question) -> vector. Avoids re-embedding the same
                     # question once per config, which would multiply the API bill.

def _sweep_embed_query(cfg, question):
    key = (cfg["model"], cfg["dim"], question)
    if key not in _sweep_qcache:
        _sweep_qcache[key] = embed_texts([question], "RETRIEVAL_QUERY",
                                         model=cfg["model"], dim=cfg["dim"])[0]
    return _sweep_qcache[key]

def _sweep_table_name(cfg):
    """Turn a config name into a legal, unique Postgres table name."""
    slug = re.sub(r"[^a-z0-9]+", "_", cfg["name"].lower()).strip("_")
    return f"rag_sweep_{slug}"[:60]        # Postgres identifiers max out at 63 characters

def sweep_ingest(cfg):
    """Chunk + embed the whole corpus under ONE configuration, into its own table."""
    table = _sweep_table_name(cfg)
    with db() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {table}")     # safe: these are throwaway tables
        cur.execute(f"""
            CREATE TABLE {table} (
                chunk_id  bigserial PRIMARY KEY,
                doc_id    text NOT NULL,
                content   text NOT NULL,
                embedding vector({cfg['dim']}) NOT NULL
            )
        """)

    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT doc_id, title, content, metadata FROM {DOC_TABLE} ORDER BY doc_id")
        docs = cur.fetchall()

    chunk_fn = CHUNKERS[cfg["chunker"]]
    rows = []                       # (doc_id, clean_text, text_we_embed)
    for d in docs:
        header = contextual_header(d["doc_id"], d["title"], d["metadata"]) if cfg["header"] else ""
        for piece in chunk_fn(d["content"]):
            rows.append((d["doc_id"], piece, f"{header}\n\n{piece}" if header else piece))

    vectors = embed_texts([r[2] for r in rows], "RETRIEVAL_DOCUMENT",
                          model=cfg["model"], dim=cfg["dim"])

    with db() as cur:
        execute_values(cur,
                       f"INSERT INTO {table} (doc_id, content, embedding) VALUES %s",
                       [(r[0], r[1], str(v)) for r, v in zip(rows, vectors)],
                       template="(%s, %s, %s::vector)", page_size=200)
        # HNSW cannot index beyond 2000 dims - the 3072 config falls back to an exact scan.
        # That is fine for a sweep (we are measuring retrieval QUALITY, not speed) but it is
        # exactly the constraint that makes 3072 impractical to actually deploy.
        if cfg["dim"] <= 2000:
            cur.execute(f"CREATE INDEX ON {table} USING hnsw (embedding vector_cosine_ops)")
        cur.execute(f"ANALYZE {table}")
    return table, len(rows)

def sweep_eval(cfg, table, k=RERANK_TOP_N):
    """Score one configuration on the GOLD questions using dense retrieval only.

    Vector-only on purpose: the keyword leg is identical for every config (it does not use
    embeddings), so including it would dilute exactly the difference we are trying to measure.
    """
    answerable = [g for g in GOLD if not g.get("unanswerable")]
    recall = mrr = support = 0.0
    for g in answerable:
        qv = _sweep_embed_query(cfg, g["q"])
        with db(dict_rows=True) as cur:
            cur.execute(f"""
                SELECT doc_id, content, embedding <=> %s::vector AS distance
                FROM {table} ORDER BY distance LIMIT %s
            """, (str(list(qv)), k))
            hits = [dict(r) for r in cur.fetchall()]

        ranked = [h["doc_id"] for h in hits]
        gold_docs = set(g["doc_ids"])
        if gold_docs & set(ranked):
            recall += 1
            mrr += 1.0 / (next(i for i, d in enumerate(ranked) if d in gold_docs) + 1)
        blob = " ".join(h["content"] for h in hits).lower()
        if all(s.lower() in blob for s in g["must_contain"]):
            support += 1            # was the answer TEXT actually retrieved, not just the doc?

    n = len(answerable)
    return {"recall": recall / n, "mrr": mrr / n, "support": support / n, "n": n}

def wilson_ci(p, n, z=1.96):
    """95% confidence interval for a proportion (Wilson score interval).

    Plain-English version: 'the true score is somewhere in this range'. Wilson is used
    instead of the textbook formula because it stays sensible for small n and for scores
    near 0 or 1 - exactly our situation. Returns (low, high).
    """
    if n == 0:
        return (0.0, 0.0)
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))

In [ ]:
# =====================================================================================
# Run the sweep. Set CONFIGS_TO_RUN = SWEEP_CONFIGS[:4] first if you want a cheap trial.
# =====================================================================================
CONFIGS_TO_RUN = SWEEP_CONFIGS

sweep_results, sweep_tables = [], []
for cfg in CONFIGS_TO_RUN:
    try:
        t0 = time.perf_counter()
        table, n_chunks = sweep_ingest(cfg)
        ingest_s = time.perf_counter() - t0
        sweep_tables.append(table)

        scores = sweep_eval(cfg, table)
        # pgvector stores a vector as 4 bytes per dimension plus ~8 bytes of row overhead.
        mb = n_chunks * (4 * cfg["dim"] + 8) / 1024 / 1024
        sweep_results.append({**cfg, **scores, "chunks": n_chunks,
                              "ingest_s": ingest_s, "vector_mb": mb, "ok": True})
        print(f"  ok   {cfg['name']:<42} {n_chunks:>4} chunks  {ingest_s:>5.1f}s")
    except Exception as e:
        # A config can legitimately fail - a model not enabled in your project, a dimension
        # the model does not support. Record it and keep going rather than losing the run.
        sweep_results.append({**cfg, "ok": False, "error": f"{type(e).__name__}: {e}"})
        print(f"  FAIL {cfg['name']:<42} {str(e)[:60]}")

# ---------------------------------- report ----------------------------------
ok = [r for r in sweep_results if r["ok"]]
ok.sort(key=lambda r: (r["support"], r["mrr"], r["recall"]), reverse=True)

hdr = (f"{'configuration':<44}{'recall':>8}{'mrr':>7}{'support':>9}"
       f"{'95% CI (support)':>20}{'chunks':>8}{'vec MB':>9}")
print("\n" + hdr)
print("-" * len(hdr))
for r in ok:
    lo, hi = wilson_ci(r["support"], r["n"])
    print(f"{r['name']:<44}{r['recall']:>8.2f}{r['mrr']:>7.2f}{r['support']:>9.2f}"
          f"{f'[{lo:.2f}, {hi:.2f}]':>20}{r['chunks']:>8}{r['vector_mb']:>9.2f}")

for r in sweep_results:
    if not r["ok"]:
        print(f"  skipped: {r['name']} -> {r['error'][:90]}")

# ------------------------- the honesty check on your own result -------------------------
if ok:
    best = ok[0]
    b_lo, b_hi = wilson_ci(best["support"], best["n"])
    # Anything whose interval overlaps the winner's cannot be called worse on this evidence.
    tied = [r for r in ok[1:] if wilson_ci(r["support"], r["n"])[1] >= b_lo]
    print(f"\nHighest score: {best['name']} (answer-support {best['support']:.2f})")
    print(f"Statistically indistinguishable from it on {best['n']} questions: {len(tied)} config(s)")
    if tied:
        for r in tied[:5]:
            print(f"    - {r['name']}")
        print(f"\n  With only {best['n']} questions the confidence interval is ~"
              f"±{(b_hi - b_lo) / 2 * 100:.0f} points, so this sweep CANNOT separate those.")
        print("  Report it as directional. To actually rank them, grow GOLD to 50-100 questions.")
    else:
        print("  This config beats every other tested config beyond the confidence interval.")

# ------------------------------- clean up the temp tables -------------------------------
# These really are disposable - unlike v1.1's Step 12, which dropped the winning table.
# Comment this out if you want to inspect a sweep table by hand afterwards.
with db() as cur:
    for t in sweep_tables:
        cur.execute(f"DROP TABLE IF EXISTS {t}")
print(f"\nDropped {len(sweep_tables)} temporary sweep table(s). "
      f"Production tables ({DOC_TABLE}, {CHUNK_TABLE}) untouched.")

## Step 14 — Production checklist

What changes between this notebook and a deployed service.

### Data & indexing
- **Re-index on write, not on a schedule.** Call `upsert_documents()` from whatever writes your source
  data, then `ingest()`. The content-hash check makes it cheap and safe to call often.
- **Bulk backfill:** load rows first, build HNSW last (`ensure_vector_index()` already assumes that
  order). Raise `maintenance_work_mem` (e.g. `SET maintenance_work_mem = '2GB'`) for the build — an
  HNSW build that spills to disk is dramatically slower.
- **`ANALYZE` after large loads** so the planner keeps choosing the index.
- Watch re-ingest churn: `pg_stat_user_tables.n_dead_tup`, and tune autovacuum on `rag_chunks`.
- **Metadata filters + ANN interact badly at scale.** HNSW filters *after* the scan, so a narrow filter
  can return fewer rows than you asked for. On pgvector ≥ 0.8 set `hnsw.iterative_scan = relaxed_order`;
  otherwise raise `hnsw.ef_search` when filtering, or use a partial index per high-traffic filter value.

### Retrieval tuning, in the order that pays
1. `hnsw.ef_search` — the recall/latency dial. Start at 100, raise until recall plateaus. Free to change.
2. `CHUNK_TOKENS` / `CHUNK_OVERLAP_TOKENS` — needs a re-ingest; measure with Step 11 before and after.
3. `RRF_K`, `W_VECTOR`/`W_TEXT` — push lexical weight up if your users search by ID, SKU or code.
4. `m` / `ef_construction` — needs an index rebuild. Only after 1–3 are exhausted.

### Reliability
- Swap the LLM reranker for the **Vertex AI Ranking API** (cheaper, faster, purpose-built). `rerank()` is the seam.
- Vertex embedding quota is the usual first bottleneck. `_with_retry` absorbs bursts; for backfills add a
  bounded worker pool and a rate limiter.
- Set request timeouts and a circuit breaker around Vertex. A retrieval-only degraded mode (return the
  sources, skip generation) beats a 30-second hang.
- Keep the pool smaller than Cloud SQL's `max_connections` divided by your instance count.

### Observability — log every request
`ask()` already returns per-stage timings and the retrieved `doc_id`s. Persist them. Four signals tell
you retrieval is degrading before users complain:
`refusal rate`, `p95 search_ms`, `mean top-1 cosine_distance`, `% of answers with zero citations`.

### Correctness
- **Grow the gold set to 50–100 questions from real user logs**, and run Steps 11 + 12 in CI. Treat a
  regression in `answer_support` as a build failure.
- Re-run the eval after *any* change to chunking, embedding model, `EMBED_DIM`, or the prompt.
- Embedding models are **not** interchangeable: changing `EMBED_MODEL` or `EMBED_DIM` invalidates every
  stored vector. Re-embed the whole corpus with `ingest(force=True)` — a mixed-model table returns nonsense.

### Security
- **Rotate the Postgres password that was hardcoded in v1.1** if that file ever left your machine.
- Use IAM database authentication or Secret Manager; never a literal in a notebook.
- If documents are access-controlled, filter by permission **in the SQL** (`metadata @> ...` on a
  tenant/ACL key) — never by post-filtering the LLM's output.

### Worth adding next
- **Query rewriting** — resolve pronouns and follow-ups against conversation history before retrieval.
- **Full contextual retrieval** — an LLM-written 1–2 sentence situating summary per chunk (a paid upgrade
  over the free header used here).
- **Parent-document retrieval** — search small chunks, feed the surrounding section to the LLM.
- **Semantic caching** on the query embedding for repeated questions.

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `column cannot have more than 2000 dimensions` | `EMBED_DIM > 2000`. Keep 1536 (or 768). This is what forced v1.1 into brute-force scans. |
| `403 PERMISSION_DENIED` from Vertex | `gcloud auth application-default login`, and enable the Vertex AI API on the project. |
| `429 RESOURCE_EXHAUSTED` during ingest | `_with_retry` backs off automatically; for large backfills lower `batch_size` and add a rate limiter. |
| `type "vector" does not exist` | Step 4 ran against a different database than `retrieve()` uses. Check `PGDATABASE`. |
| `unrecognized configuration parameter "hnsw.ef_search"` | pgvector too old, or the extension is not loaded in this session. Check `SELECT extversion FROM pg_extension WHERE extname='vector'`. |
| Search returns nothing for an exact ID | The lexical leg needs that term in `embed_input`. Check with `SELECT embed_input FROM rag_chunks WHERE doc_id = '...'`. |
| Answers are right but slow | Look at `ask()["timings"]` — it is almost always `generate_ms` or `rerank_ms`, never `search_ms`. |
| Right document cited, wrong number stated | Rules 2/3 in `SYSTEM_RULES` — the exact v1.1 failure. Confirm the header is present in `embed_input`, and raise `RERANK_TOP_N` so competing values are both in context. |
| `recall` high but `support` low | Chunks too small, or the answer straddles a boundary. Raise `CHUNK_TOKENS` / `CHUNK_OVERLAP_TOKENS` and re-ingest. |
| Everything got worse after a model change | Stored vectors are stale. `ingest(force=True)`. |

### Teardown (only if you want to discard the index)

Unlike v1.1's Step 12, this does **not** run by default — v1.1 dropped the very table it had just crowned
the winner.

```python
# with db() as cur:
#     cur.execute(f"DROP TABLE IF EXISTS {CHUNK_TABLE}, {DOC_TABLE} CASCADE")
```